**Cell 1: Install dependencies and import libraries**

In [1]:
# Cell 1: Install and import all required libraries
!pip install catboost xgboost lightgbm prophet statsmodels

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import random
from datetime import datetime

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from sklearn.linear_model import LinearRegression

print("✅ All dependencies installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00
✅ All dependencies installed and imported.


**Cell 2: Mount Google Drive (optional) and load CSV files**

In [2]:
# Cell 2: Mount Google Drive (if files are stored there) and load all datasets
from google.colab import drive
drive.mount('/content/drive')

# Define file paths (adjust if you store files elsewhere)
main_path = "/content/ugc_al_admission_raw.csv"
subjects_path = "/content/A_L Subjects List.csv"
aptitude_path = "/content/Aptitude Course Names, Uni Names & Uni Codes.csv"
courses_path = "/content/Course Names, Uni Names & Uni Codes.csv"
cutoffs_path = "/content/Cut-offs.csv"

# Load data
df_main = pd.read_csv(main_path)
df_subjects = pd.read_csv(subjects_path)
df_aptitude = pd.read_csv(aptitude_path)
df_courses = pd.read_csv(courses_path)
df_cutoffs = pd.read_csv(cutoffs_path)

print("✅ Data loaded successfully.")
print(f"Main admissions data: {df_main.shape}")
print(f"Subjects list: {df_subjects.shape}")
print(f"Aptitude courses: {df_aptitude.shape}")
print(f"Course-University mapping: {df_courses.shape}")
print(f"Cut-offs data: {df_cutoffs.shape}")

Mounted at /content/drive
✅ Data loaded successfully.
Main admissions data: (3768, 33)
Subjects list: (62, 3)
Aptitude courses: (24, 3)
Course-University mapping: (261, 3)
Cut-offs data: (28825, 8)


**Cell 3: Build course‑to‑university mapping and aptitude set**

In [3]:
# Cell 3: Create course‑to‑university mapping and set of aptitude‑required courses

aptitude_courses_set = set()
for _, row in df_aptitude.iterrows():
    course = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
    if course:
        aptitude_courses_set.add(course.upper())

print(f"✅ Loaded {len(aptitude_courses_set)} aptitude‑required courses.")

course_uni_mapping = {}
for _, row in df_courses.iterrows():
    course = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
    university = str(row.iloc[1]).strip() if len(row) > 1 and pd.notna(row.iloc[1]) else ""
    uni_code = str(row.iloc[2]).strip() if len(row) > 2 and pd.notna(row.iloc[2]) else ""
    if course:
        course_upper = course.upper()
        course_uni_mapping[course_upper] = {
            "Course": course,
            "University": university,
            "Uni_Code": uni_code,
            "Aptitude": "Yes" if course_upper in aptitude_courses_set else "No"
        }

print(f"✅ Created mapping for {len(course_uni_mapping)} courses.")

✅ Loaded 15 aptitude‑required courses.
✅ Created mapping for 88 courses.


**Cell 4: Build stream eligibility map (manually curated from your provided data)**

In [4]:
# Cell 4: Manually defined stream‑eligibility map (based on your final message)
STREAM_ELIGIBILITY_MAP_DECODED = {
    "Arts": [
        ("Arts", "019A", "University of Colombo", "No"),
        ("Arts", "019B", "University of Peradeniya", "No"),
        ("Arts", "019C", "University of Sri Jayewardenepura", "No"),
        ("Arts", "019D", "University of Kelaniya", "No"),
        ("Arts", "019E", "University of Jaffna", "No"),
        ("Arts", "019F", "University of Ruhuna", "No"),
        ("Arts", "019H", "Eastern University, Sri Lanka", "No"),
        ("Arts", "019J", "South Eastern University of Sri Lanka", "No"),
        ("Arts", "019K", "Rajarata University of Sri Lanka", "No"),
        ("Arts (SP) - Mass Media", "020S", "Sripalee Campus, University of Colombo", "Yes"),
        ("Arts (SP) - Performing Arts", "041S", "Sripalee Campus, University of Colombo", "Yes"),
        ("Arts (SAB)", "021L", "Sabaragamuwa University of Sri Lanka", "No"),
        ("Communication Studies", "029W", "Trincomalee Campus, Eastern University", "No"),
        ("Peace & Conflict Resolution", "031D", "University of Kelaniya", "No"),
        ("Islamic Studies", "063J", "South Eastern University of Sri Lanka", "No"),
        ("Arabic Language", "084J", "South Eastern University of Sri Lanka", "No"),
        ("Teaching English as a Second Language (TESL)", "105C", "University of Sri Jayewardenepura", "No"),
        ("Teaching English as a Second Language (TESL)", "105D", "University of Kelaniya", "No"),
        ("Teaching English as a Second Language (TESL)", "105L", "Sabaragamuwa University of Sri Lanka", "No"),
        ("Social Work", "112B", "University of Peradeniya", "No"),
        ("Social Work", "112C", "University of Sri Jayewardenepura", "No"),
        ("Arts - Information Technology", "128C", "University of Sri Jayewardenepura", "No")
    ],
    "Commerce": [
        ("Management", "016A", "University of Colombo", "No"),
        ("Management", "016B", "University of Peradeniya", "No"),
        ("Management", "016C", "University of Sri Jayewardenepura", "No"),
        ("Management", "016D", "University of Kelaniya", "No"),
        ("Management", "016E", "University of Jaffna", "No"),
        ("Management", "016F", "University of Ruhuna", "No"),
        ("Management", "016H", "Eastern University", "No"),
        ("Management", "016J", "South Eastern University", "No"),
        ("Management", "016K", "Rajarata University", "No"),
        ("Management", "016L", "Sabaragamuwa University", "No"),
        ("Management", "016M", "Wayamba University", "No"),
        ("Management and Public Policy", "028C", "University of Sri Jayewardenepura", "No"),
        ("Real Estate Management and Valuation", "017C", "University of Sri Jayewardenepura", "No"),
        ("Commerce", "018C", "University of Sri Jayewardenepura", "No"),
        ("Commerce", "018D", "University of Kelaniya", "No"),
        ("Commerce", "018E", "University of Jaffna", "No"),
        ("Commerce", "018H", "Eastern University", "No"),
        ("Commerce", "018J", "South Eastern University", "No"),
        ("Management Studies (TV)", "022W", "Trincomalee Campus", "No"),
        ("Management Studies (TV)", "022R", "University of Vavuniya", "No"),
        ("Business Information Systems (Honours) (BIS)", "077C", "University of Sri Jayewardenepura", "No"),
        ("Accounting Information Systems", "127D", "University of Kelaniya", "No"),
        ("Banking and Insurance", "133R", "University of Vavuniya", "No"),
        ("Service Management", "140P", "Gampaha Wickramarachchi University", "No")
    ],
    "Biological Science": [
        ("Medicine", "001A", "University of Colombo", "No"),
        ("Medicine", "001B", "University of Peradeniya", "No"),
        ("Medicine", "001C", "University of Sri Jayewardenepura", "No"),
        ("Medicine", "001D", "University of Kelaniya", "No"),
        ("Medicine", "001E", "University of Jaffna", "No"),
        ("Medicine", "001F", "University of Ruhuna", "No"),
        ("Medicine", "001G", "University of Moratuwa", "No"),
        ("Medicine", "001H", "Eastern University", "No"),
        ("Medicine", "001K", "Rajarata University", "No"),
        ("Medicine", "001L", "Sabaragamuwa University", "No"),
        ("Medicine", "001M", "Wayamba University", "No"),
        ("Medicine", "001U", "Uva Wellassa University", "No"),
        ("Dental Surgery", "002B", "University of Peradeniya", "No"),
        ("Dental Surgery", "002C", "University of Sri Jayewardenepura", "No"),
        ("Veterinary Science", "003B", "University of Peradeniya", "No"),
        ("Agriculture", "004E", "University of Jaffna", "No"),
        ("Agriculture", "004H", "Eastern University", "No"),
        ("Agriculture", "004K", "Rajarata University", "No"),
        ("Agriculture", "004L", "Sabaragamuwa University", "No"),
        ("Agriculture", "004M", "Wayamba University", "No"),
        ("Food Science & Nutrition", "005M", "Wayamba University", "No"),
        ("Biological Science", "006A", "University of Colombo", "No"),
        ("Biological Science", "006B", "University of Peradeniya", "No"),
        ("Biological Science", "006C", "University of Sri Jayewardenepura", "No"),
        ("Biological Science", "006D", "University of Kelaniya", "No"),
        ("Biological Science", "006E", "University of Jaffna", "No"),
        ("Biological Science", "006F", "University of Ruhuna", "No"),
        ("Biological Science", "006H", "Eastern University", "No"),
        ("Biological Science", "006J", "South Eastern University", "No"),
        ("Applied Sciences (Biological Sc.)", "007K", "Rajarata University", "No"),
        ("Applied Sciences (Biological Sc.)", "007L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Biological Sc.)", "007R", "University of Vavuniya", "No"),
        ("Ayurveda Medicine and Surgery", "032A", "University of Colombo", "No"),
        ("Ayurveda Medicine and Surgery", "032P", "Gampaha Wickramarachchi University", "No"),
        ("Unani Medicine and Surgery", "033A", "University of Colombo", "No"),
        ("Siddha Medicine and Surgery", "036E", "University of Jaffna", "No"),
        ("Siddha Medicine and Surgery", "036W", "Trincomalee Campus", "No"),
        ("Nursing", "037A", "University of Colombo", "No"),
        ("Nursing", "037B", "University of Peradeniya", "No"),
        ("Nursing", "037C", "University of Sri Jayewardenepura", "No"),
        ("Nursing", "037E", "University of Jaffna", "No"),
        ("Nursing", "037F", "University of Ruhuna", "No"),
        ("Nursing", "037H", "Eastern University", "No"),
        ("Pharmacy", "051B", "University of Peradeniya", "No"),
        ("Pharmacy", "051C", "University of Sri Jayewardenepura", "No"),
        ("Pharmacy", "051E", "University of Jaffna", "No"),
        ("Pharmacy", "051F", "University of Ruhuna", "No"),
        ("Medical Laboratory Sciences", "052B", "University of Peradeniya", "No"),
        ("Medical Laboratory Sciences", "052C", "University of Sri Jayewardenepura", "No"),
        ("Medical Laboratory Sciences", "052E", "University of Jaffna", "No"),
        ("Medical Laboratory Sciences", "052F", "University of Ruhuna", "No"),
        ("Radiography", "053B", "University of Peradeniya", "No"),
        ("Physiotherapy", "054A", "University of Colombo", "No"),
        ("Physiotherapy", "054B", "University of Peradeniya", "No"),
        ("Health Promotion", "050K", "Rajarata University", "No")
    ],
    "Physical Science": [
        ("Engineering", "008B", "University of Peradeniya", "No"),
        ("Engineering", "008C", "University of Sri Jayewardenepura", "No"),
        ("Engineering", "008E", "University of Jaffna", "No"),
        ("Engineering", "008F", "University of Ruhuna", "No"),
        ("Engineering", "008G", "University of Moratuwa", "No"),
        ("Engineering", "008J", "South Eastern University", "No"),
        ("Engineering (EM)", "009G", "University of Moratuwa", "No"),
        ("Engineering (TM)", "010G", "University of Moratuwa", "No"),
        ("Quantity Surveying", "011G", "University of Moratuwa", "No"),
        ("Computer Science", "012C", "University of Sri Jayewardenepura", "No"),
        ("Computer Science", "012D", "University of Kelaniya", "No"),
        ("Computer Science", "012E", "University of Jaffna", "No"),
        ("Computer Science", "012F", "University of Ruhuna", "No"),
        ("Computer Science", "012T", "University of Colombo School of Computing", "No"),
        ("Computer Science", "012W", "Trincomalee Campus", "No"),
        ("Physical Science", "013A", "University of Colombo", "No"),
        ("Physical Science", "013B", "University of Peradeniya", "No"),
        ("Physical Science", "013C", "University of Sri Jayewardenepura", "No"),
        ("Physical Science", "013D", "University of Kelaniya", "No"),
        ("Physical Science", "013E", "University of Jaffna", "No"),
        ("Physical Science", "013F", "University of Ruhuna", "No"),
        ("Physical Science", "013H", "Eastern University", "No"),
        ("Physical Science", "013J", "South Eastern University", "No"),
        ("Surveying Science", "014L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Physical Sc.)", "015K", "Rajarata University", "No"),
        ("Applied Sciences (Physical Sc.)", "015L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Physical Sc.)", "015M", "Wayamba University", "No"),
        ("Applied Sciences (Physical Sc.)", "015R", "University of Vavuniya", "No"),
        ("Applied Sciences (Physical Sc.)", "015W", "Trincomalee Campus", "No")
    ],
    "Engineering Technology": [
        ("Engineering Technology (ET)", "102A", "University of Colombo", "No"),
        ("Engineering Technology (ET)", "102C", "University of Sri Jayewardenepura", "No"),
        ("Engineering Technology (ET)", "102D", "University of Kelaniya", "No"),
        ("Engineering Technology (ET)", "102E", "University of Jaffna", "No"),
        ("Engineering Technology (ET)", "102F", "University of Ruhuna", "No"),
        ("Engineering Technology (ET)", "102K", "Rajarata University", "No"),
        ("Engineering Technology (ET)", "102L", "Sabaragamuwa University", "No"),
        ("Engineering Technology (ET)", "102M", "Wayamba University", "No"),
        ("Engineering Technology (ET)", "102U", "Uva Wellassa University", "No")
    ],
    "Biosystems Technology": [
        ("Biosystems Technology (BST)", "103A", "University of Colombo", "No"),
        ("Biosystems Technology (BST)", "103C", "University of Sri Jayewardenepura", "No"),
        ("Biosystems Technology (BST)", "103D", "University of Kelaniya", "No"),
        ("Biosystems Technology (BST)", "103E", "University of Jaffna", "No"),
        ("Biosystems Technology (BST)", "103F", "University of Ruhuna", "No"),
        ("Biosystems Technology (BST)", "103H", "Eastern University", "No"),
        ("Biosystems Technology (BST)", "103J", "South Eastern University", "No"),
        ("Biosystems Technology (BST)", "103K", "Rajarata University", "No"),
        ("Biosystems Technology (BST)", "103L", "Sabaragamuwa University", "No"),
        ("Biosystems Technology (BST)", "103M", "Wayamba University", "No"),
        ("Biosystems Technology (BST)", "103U", "Uva Wellassa University", "No")
    ],
    "Common / Multi-Stream": [
        ("Information Technology (IT)", "026G", "University of Moratuwa", "No"),
        ("Management and Information Technology (MIT)", "027D", "University of Kelaniya", "No"),
        ("Quantity Surveying", "011G", "University of Moratuwa", "No"),
        ("Surveying Science", "014L", "Sabaragamuwa University", "No"),
        ("Urban Informatics and Planning", "030G", "University of Moratuwa", "No"),
        ("Architecture", "023G", "University of Moratuwa", "Yes"),
        ("Fashion Design & Product Development", "034G", "University of Moratuwa", "Yes"),
        ("Landscape Architecture", "097G", "University of Moratuwa", "Yes"),
        ("Design", "024G", "University of Moratuwa", "Yes"),
        ("Law", "025A", "University of Colombo", "Yes"),
        ("Law", "025B", "University of Peradeniya", "Yes"),
        ("Law", "025E", "University of Jaffna", "Yes"),
        ("Facilities Management", "056G", "University of Moratuwa", "No"),
        ("Management and Information Technology (SEUSL)", "079J", "South Eastern University", "No"),
        ("Science and Technology", "064U", "Uva Wellassa University", "No"),
        ("Computer Science & Technology", "065U", "Uva Wellassa University", "No"),
        ("Entrepreneurship and Management", "066U", "Uva Wellassa University", "No"),
        ("Industrial Information Technology", "075U", "Uva Wellassa University", "No"),
        ("Mineral Resources and Technology", "076U", "Uva Wellassa University", "No"),
        ("Hospitality, Tourism and Events Management", "090U", "Uva Wellassa University", "No"),
        ("Physical Education", "081E", "University of Jaffna", "Yes"),
        ("Physical Education", "081L", "Sabaragamuwa University", "Yes"),
        ("Sports Science & Management", "082C", "University of Sri Jayewardenepura", "Yes"),
        ("Sports Science & Management", "082D", "University of Kelaniya", "Yes"),
        ("Sports Science & Management", "082L", "Sabaragamuwa University", "Yes"),
        ("Information Technology & Management", "091G", "University of Moratuwa", "No"),
        ("Tourism & Hospitality Management", "092K", "Rajarata University", "No"),
        ("Tourism & Hospitality Management", "092L", "Sabaragamuwa University", "No"),
        ("Agricultural Resource Management and Technology", "093F", "University of Ruhuna", "No"),
        ("Agribusiness Management", "094F", "University of Ruhuna", "No"),
        ("Green Technology", "095F", "University of Ruhuna", "No"),
        ("Information Systems", "096C", "University of Sri Jayewardenepura", "No"),
        ("Information Systems", "096L", "Sabaragamuwa University", "No"),
        ("Information Systems", "096T", "UCSC", "No"),
        ("Translation Studies", "098D", "University of Kelaniya", "No"),
        ("Translation Studies", "098E", "University of Jaffna", "No"),
        ("Translation Studies", "098H", "Eastern University", "No"),
        ("Translation Studies", "098L", "Sabaragamuwa University", "No"),
        ("Software Engineering", "099C", "University of Sri Jayewardenepura", "No"),
        ("Software Engineering", "099D", "University of Kelaniya", "No"),
        ("Software Engineering", "099L", "Sabaragamuwa University", "No"),
        ("Film & Television Studies", "100D", "University of Kelaniya", "Yes"),
        ("Project Management", "101R", "University of Vavuniya", "No"),
        ("Data Science", "136L", "Sabaragamuwa University", "No"),
        ("Primary Education", "137A", "University of Colombo", "No"),
        ("Medical Imaging Technology", "138A", "University of Colombo", "No"),
        ("Polymer Science and Industrial Management", "139C", "University of Sri Jayewardenepura", "No"),
        ("Service Management", "140P", "Gampaha Wickramarachchi University", "No")
    ]
}

# Convert to set of course names per stream for quick lookup
stream_course_names = {}
for stream, courses in STREAM_ELIGIBILITY_MAP_DECODED.items():
    stream_course_names[stream] = set(c[0] for c in courses)

print("✅ Stream eligibility map created.")
for stream, names in stream_course_names.items():
    print(f"  {stream}: {len(names)} courses")

✅ Stream eligibility map created.
  Arts: 11 courses
  Commerce: 9 courses
  Biological Science: 16 courses
  Physical Science: 8 courses
  Engineering Technology: 1 courses
  Biosystems Technology: 1 courses
  Common / Multi-Stream: 35 courses


**Cell 5: Preprocess cut‑off data to create course‑level difficulty features**

In [5]:
# Cell 5: Compute cut‑off statistics per (Course, University) pair
cutoff_df = df_cutoffs.copy()
# Convert Zscore column to numeric, coercing errors to NaN
cutoff_df['Zscore'] = pd.to_numeric(cutoff_df['Zscore'], errors='coerce')
# Drop rows where Zscore is NaN (NQC entries)
cutoff_df = cutoff_df.dropna(subset=['Zscore'])

# Clean column names (strip whitespace)
cutoff_df.columns = cutoff_df.columns.str.strip()

cutoff_stats = cutoff_df.groupby(['Course', 'University']).agg(
    cutoff_min=('Zscore', 'min'),
    cutoff_max=('Zscore', 'max'),
    cutoff_mean=('Zscore', 'mean'),
    cutoff_std=('Zscore', 'std')
).reset_index()

# Create a lookup dictionary for fast access during recommendation
cutoff_lookup = {}
for _, row in cutoff_stats.iterrows():
    key = (row['Course'].strip().upper(), row['University'].strip().upper())
    cutoff_lookup[key] = {
        'mean': row['cutoff_mean'],
        'min': row['cutoff_min'],
        'max': row['cutoff_max'],
        'std': row['cutoff_std']
    }

print(f"✅ Cut‑off statistics computed for {len(cutoff_lookup)} course‑university pairs.")

✅ Cut‑off statistics computed for 383 course‑university pairs.


**Cell 6: Preprocess the main admissions dataframe**

In [6]:
# Cell 6: Clean and impute the main dataset
df = df_main.copy()
print("Initial shape:", df.shape)

# Z‑Score bounds (realistic range)
df["Z_Score"] = pd.to_numeric(df["Z_Score"], errors='coerce')
df["Z_Score"] = df["Z_Score"].clip(-1.5, 4.0).fillna(df["Z_Score"].median())

# Numerical columns imputation
num_cols = ["Z_Score", "Island_Rank", "Gen_Test"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

# Categorical columns imputation
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

# Grade validation
valid_grades = ["A", "B", "C", "S", "F"]
for g in ["Grade_1", "Grade_2", "Grade_3"]:
    if g in df.columns:
        df[g] = df[g].where(df[g].isin(valid_grades), "S")

# Add aptitude requirement from mapping
def get_aptitude(course_name):
    if pd.isna(course_name):
        return "No"
    course_upper = str(course_name).upper()
    return "Yes" if course_upper in aptitude_courses_set else "No"

df["aptitude_required"] = df["Course"].apply(get_aptitude)

print("✅ Data cleaning completed.")
print("Aptitude distribution:")
print(df["aptitude_required"].value_counts())

Initial shape: (3768, 33)
✅ Data cleaning completed.
Aptitude distribution:
aptitude_required
No     3123
Yes     645
Name: count, dtype: int64


**Cell 7: Feature engineering – grade numeric and interest clusters**

In [7]:
# Cell 7: Create numeric grade features and interest clusters
grade_mapping = {"A": 4, "B": 3, "C": 2, "S": 1, "F": 0}
for g in ["Grade_1", "Grade_2", "Grade_3"]:
    if g in df.columns:
        df[f"{g}_num"] = df[g].map(grade_mapping).fillna(1)

grade_cols = [f"{g}_num" for g in ["Grade_1", "Grade_2", "Grade_3"] if f"{g}_num" in df.columns]
if grade_cols:
    df["Avg_Grade"] = df[grade_cols].mean(axis=1)
else:
    df["Avg_Grade"] = 2.0

# Career interest questions (1‑5 scale)
interest_cols = ['q1_science_tech', 'q2_healthcare', 'q3_design', 'q4_data',
                 'q5_business', 'q6_arts_culture', 'q7_nature_env',
                 'q8_hands_on', 'q9_innovation', 'q10_people_social',
                 'q11_urban_corporate', 'q12_flexible_path']

for col in interest_cols:
    if col not in df.columns:
        df[col] = 3
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(3).clip(1, 5)

# Interest clusters
df["STEM_Interest"] = (df["q1_science_tech"] + df["q4_data"] + df["q9_innovation"]) / 3
df["Healthcare_Interest"] = (df["q2_healthcare"] + df["q10_people_social"]) / 2
df["Creative_Interest"] = (df["q3_design"] + df["q6_arts_culture"]) / 2
df["Nature_Interest"] = (df["q7_nature_env"] + df["q8_hands_on"]) / 2
df["Business_Interest"] = (df["q5_business"] + df["q11_urban_corporate"] + df["q12_flexible_path"]) / 3

print("✅ Feature engineering completed. Total columns:", len(df.columns))

✅ Feature engineering completed. Total columns: 42


**Cell 8: Generate time‑series features (trend, smoothing, seasonal) per course**

In [8]:
# Cell 8: Time‑series features using simple methods (avoid heavy ARIMA/Prophet)
def generate_trend_features(group_df):
    if len(group_df) < 3:
        return {'trend_slope': 0, 'forecast': group_df['Z_Score'].mean()}
    group_df = group_df.sort_values('Year')
    years = group_df['Year'].values.reshape(-1,1)
    z = group_df['Z_Score'].values
    try:
        lr = LinearRegression().fit(years, z)
        slope = lr.coef_[0]
        forecast = lr.predict([[years[-1,0]+1]])[0]
        return {'trend_slope': slope, 'forecast': forecast}
    except:
        return {'trend_slope': 0, 'forecast': z[-1]}

def generate_smoothing_features(group_df):
    if len(group_df) < 3:
        return {'smoothed_value': group_df['Z_Score'].mean(), 'smoothed_forecast': group_df['Z_Score'].mean()}
    group_df = group_df.sort_values('Year')
    z = group_df['Z_Score'].values
    try:
        model = SimpleExpSmoothing(z).fit(smoothing_level=0.3, optimized=False)
        smoothed = model.fittedvalues
        forecast = model.forecast(1).iloc[0]
        return {'smoothed_value': smoothed[-1], 'smoothed_forecast': forecast}
    except:
        return {'smoothed_value': z[-1], 'smoothed_forecast': z[-1]}

def generate_seasonal_features(group_df):
    if len(group_df) < 4:
        return {'seasonal_strength': 0, 'year_over_year_change': 0, 'rolling_mean_3yr': group_df['Z_Score'].mean()}
    group_df = group_df.sort_values('Year')
    z = group_df['Z_Score'].values
    yoy_changes = [z[i]-z[i-1] for i in range(1, len(z))]
    avg_yoy = np.mean(yoy_changes) if yoy_changes else 0
    rolling_mean = np.mean(z[-3:]) if len(z)>=3 else np.mean(z)
    seasonal_strength = np.std(yoy_changes) if len(yoy_changes)>1 else 0
    return {'seasonal_strength': seasonal_strength, 'year_over_year_change': avg_yoy, 'rolling_mean_3yr': rolling_mean}

# Ensure Year is numeric
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

course_ids = df['Course'].unique()
trend_list = []
for cid in course_ids:
    group = df[df['Course']==cid][['Year','Z_Score']].dropna()
    if len(group)>=3:
        trend = generate_trend_features(group)
        smooth = generate_smoothing_features(group)
        seasonal = generate_seasonal_features(group)
        trend_list.append({
            'Course': cid,
            'trend_slope': trend['trend_slope'],
            'trend_forecast': trend['forecast'],
            'smoothed_value': smooth['smoothed_value'],
            'smoothed_forecast': smooth['smoothed_forecast'],
            'seasonal_strength': seasonal['seasonal_strength'],
            'year_over_year_change': seasonal['year_over_year_change'],
            'rolling_mean_3yr': seasonal['rolling_mean_3yr']
        })

trend_df = pd.DataFrame(trend_list)
if not trend_df.empty:
    df = df.merge(trend_df, on='Course', how='left')
    fill_cols = ['trend_slope','trend_forecast','smoothed_value','smoothed_forecast',
                 'seasonal_strength','year_over_year_change','rolling_mean_3yr']
    for col in fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df['Z_Score'].mean())
else:
    for col in ['trend_slope','trend_forecast','smoothed_value','smoothed_forecast',
                'seasonal_strength','year_over_year_change','rolling_mean_3yr']:
        df[col] = 0

print("✅ Time‑series features added.")

✅ Time‑series features added.


**Cell 9: Create forecast features (ARIMA, simple forecast, hybrid)**

In [9]:
# Cell 9: Simple forecast (moving average) as proxy for Prophet
def simple_forecast(group_df):
    if len(group_df) < 2:
        return np.nan
    group_df = group_df.sort_values('Year')
    window = min(3, len(group_df))
    return group_df['Z_Score'].tail(window).mean()

simple_forecast_values = (
    df.groupby('Course')[['Year','Z_Score']]
      .apply(lambda g: simple_forecast(g))
      .reset_index(name='Simple_Forecast_Z')
)
df = df.merge(simple_forecast_values, on='Course', how='left')
df['Simple_Forecast_Z'] = df['Simple_Forecast_Z'].fillna(df['Z_Score'].median())

# Create ARIMA_Z as simple rolling mean (3‑year)
df['ARIMA_Z'] = df.groupby('Course')['Z_Score'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().fillna(x.mean())
)

# Hybrid = average of ARIMA and Simple
df['Hybrid_Z'] = (df['ARIMA_Z'] + df['Simple_Forecast_Z']) / 2

print("✅ Forecast features (ARIMA, Simple, Hybrid) created.")

✅ Forecast features (ARIMA, Simple, Hybrid) created.


**Cell 10: Encode categorical variables and scale numerical features**

In [10]:
# Cell 10: Label encoding and scaling
label_encoders = {}
cat_cols_enc = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols_enc:
    le = LabelEncoder()
    # Include 'Unknown' for future unseen values
    all_vals = df[col].astype(str).unique().tolist()
    if 'Unknown' not in all_vals:
        all_vals.append('Unknown')
    le.fit(all_vals)
    df[col] = le.transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded {col}: {len(le.classes_)} classes")

# Numerical columns to scale
num_to_scale = ["Z_Score", "Island_Rank", "Gen_Test", "Avg_Grade",
                "ARIMA_Z", "Simple_Forecast_Z", "Hybrid_Z",
                "STEM_Interest", "Healthcare_Interest", "Creative_Interest",
                "Nature_Interest", "Business_Interest",
                "trend_slope", "trend_forecast", "smoothed_value", "smoothed_forecast",
                "seasonal_strength", "year_over_year_change", "rolling_mean_3yr"] + interest_cols
num_to_scale = [c for c in num_to_scale if c in df.columns]

scaler = StandardScaler()
df[num_to_scale] = scaler.fit_transform(df[num_to_scale])

print(f"✅ Scaled {len(num_to_scale)} numerical columns. Final shape: {df.shape}")

  Encoded Stream: 7 classes
  Encoded Subject_1: 89 classes
  Encoded Grade_1: 6 classes
  Encoded Subject_2: 72 classes
  Encoded Grade_2: 6 classes
  Encoded Subject_3: 73 classes
  Encoded Grade_3: 6 classes
  Encoded District: 33 classes
  Encoded Sinhala/Tamil: 6 classes
  Encoded English: 7 classes
  Encoded Maths: 7 classes
  Encoded Science: 7 classes
  Encoded Course: 224 classes
  Encoded University: 47 classes
  Encoded Uni Code: 266 classes
  Encoded aptitude_required: 3 classes
  Encoded University Selected: 3 classes
✅ Scaled 31 numerical columns. Final shape: (3768, 52)


**Cell 11: Temporal train/validation/test split and feature set preparation**

In [11]:
# Cell 11: Split data by year (time‑aware)
df = df.sort_values("Year")
train_df = df[df["Year"] <= df["Year"].quantile(0.6)]
val_df   = df[(df["Year"] > df["Year"].quantile(0.6)) & (df["Year"] <= df["Year"].quantile(0.8))]
test_df  = df[df["Year"] > df["Year"].quantile(0.8)]

print(f"Train shape: {train_df.shape}, Val shape: {val_df.shape}, Test shape: {test_df.shape}")

target_cols = ["Course", "University", "Uni Code", "aptitude_required"]
cols_to_drop = target_cols + [c for c in df.columns if '_num' in c] + ['Year_dt'] if 'Year_dt' in df.columns else target_cols
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

X_train = train_df.drop(columns=cols_to_drop, errors='ignore')
X_val   = val_df.drop(columns=cols_to_drop, errors='ignore')
X_test  = test_df.drop(columns=cols_to_drop, errors='ignore')

y_train = train_df["Course"]
y_val   = val_df["Course"]
y_test  = test_df["Course"]

# Create feature sets by replacing Z_Score with respective forecast
X_train_arima = X_train.copy(); X_train_arima["Z_Score"] = train_df["ARIMA_Z"].values
X_val_arima   = X_val.copy();   X_val_arima["Z_Score"]   = val_df["ARIMA_Z"].values
X_test_arima  = X_test.copy();  X_test_arima["Z_Score"]  = test_df["ARIMA_Z"].values

X_train_prophet = X_train.copy(); X_train_prophet["Z_Score"] = train_df["Simple_Forecast_Z"].values
X_val_prophet   = X_val.copy();   X_val_prophet["Z_Score"]   = val_df["Simple_Forecast_Z"].values
X_test_prophet  = X_test.copy();  X_test_prophet["Z_Score"]  = test_df["Simple_Forecast_Z"].values

X_train_hybrid = X_train.copy(); X_train_hybrid["Z_Score"] = train_df["Hybrid_Z"].values
X_val_hybrid   = X_val.copy();   X_val_hybrid["Z_Score"]   = val_df["Hybrid_Z"].values
X_test_hybrid  = X_test.copy();  X_test_hybrid["Z_Score"]  = test_df["Hybrid_Z"].values

print("✅ Feature matrices prepared for ARIMA, Prophet (Simple), and Hybrid.")

Train shape: (2668, 52), Val shape: (1020, 52), Test shape: (0, 52)
✅ Feature matrices prepared for ARIMA, Prophet (Simple), and Hybrid.


**Cell 12: Define model evaluation function (handles class mismatch)**

In [12]:
# Cell 12: Evaluation function that filters validation to classes present in training
def evaluate_model_fixed(model, X_train, y_train, X_val, y_val, model_name):
    print(f"\n📊 Training {model_name}...")
    train_classes = sorted(y_train.unique())
    val_classes = sorted(y_val.unique())
    n_classes = len(train_classes)
    print(f"  - Training classes: {len(train_classes)}")
    print(f"  - Validation classes: {len(val_classes)}")
    print(f"  - Training samples: {len(X_train)}")
    print(f"  - Validation samples: {len(X_val)}")

    # Keep only validation samples whose class exists in training
    valid_mask = y_val.isin(train_classes)
    X_val_filt = X_val[valid_mask]
    y_val_filt = y_val[valid_mask]
    print(f"  - Validation samples after filtering: {len(X_val_filt)}")
    if len(X_val_filt) == 0:
        print("  ⚠️ No valid validation samples – skipping.")
        return 0.0, 0.0, None

    # Configure model for multi‑class
    model_type = str(type(model)).lower()
    if 'xgboost' in model_type:
        model.set_params(objective='multi:softprob', num_class=n_classes,
                         eval_metric='mlogloss', use_label_encoder=False, random_state=42)
        model.fit(X_train, y_train)
    elif 'lightgbm' in model_type:
        model.set_params(objective='multiclass', num_class=n_classes,
                         metric='multi_logloss', verbose=-1, random_state=42)
        model.fit(X_train, y_train, eval_set=[(X_val_filt, y_val_filt)], verbose=False)
    elif 'catboost' in model_type:
        model.set_params(loss_function='MultiClass', eval_metric='MultiClass',
                         verbose=False, random_seed=42)
        model.fit(X_train, y_train, eval_set=(X_val_filt, y_val_filt), verbose=False)

    y_pred = model.predict(X_val_filt)
    acc = accuracy_score(y_val_filt, y_pred)
    f1 = f1_score(y_val_filt, y_pred, average='weighted')
    print(f"  ✅ Accuracy: {acc:.4f}, F1: {f1:.4f}")
    return acc, f1, model

**Cell 13: Train all 9 models (ARIMA, Prophet, Hybrid × XGBoost, LightGBM, CatBoost)**

In [14]:
# Cell 13: Train and evaluate all model combinations with robust class handling

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SOLUTION 1: Custom Classifier with built-in class handling
# ============================================================

class RobustTimeSeriesClassifier:
    """
    A robust classifier wrapper that handles class mismatch between
    training and validation/test sets in time series forecasting.
    """

    def __init__(self, base_model, model_name, handle_unknown='filter'):
        """
        Parameters:
        - base_model: The underlying classifier (XGBoost, LightGBM, CatBoost)
        - model_name: Name of the model for logging
        - handle_unknown: Strategy for unknown classes in validation
                         'filter' - Remove samples with unknown classes
                         'ignore' - Keep but ignore in metrics (sets to -1)
                         'map' - Map to nearest known class (not implemented)
        """
        self.base_model = base_model
        self.model_name = model_name
        self.handle_unknown = handle_unknown
        self.label_encoder = LabelEncoder()
        self.train_classes = None
        self.is_fitted = False

    def fit(self, X, y):
        """Fit the model with label encoding"""
        print(f"    Fitting {self.model_name}...")
        print(f"      - Training classes: {len(np.unique(y))}")

        # Store training classes for later reference
        self.train_classes = np.unique(y)

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)

        # Fit the base model
        self.base_model.fit(X, y_encoded)
        self.is_fitted = True

        return self

    def predict(self, X):
        """Make predictions"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")
        return self.base_model.predict(X)

    def predict_proba(self, X):
        """Predict probabilities"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")
        return self.base_model.predict_proba(X)

    def evaluate(self, X_val, y_val, verbose=True):
        """
        Evaluate the model on validation data, handling class mismatches
        """
        if not self.is_fitted:
            raise ValueError("Model must be fitted before evaluation")

        # Find which validation samples have classes seen in training
        val_classes = np.unique(y_val)
        unknown_classes = set(val_classes) - set(self.train_classes)

        if verbose:
            print(f"    Evaluating {self.model_name}...")
            print(f"      - Training classes: {len(self.train_classes)}")
            print(f"      - Validation classes: {len(val_classes)}")
            print(f"      - Unknown classes: {len(unknown_classes)}")

        if self.handle_unknown == 'filter':
            # Filter out samples with unknown classes
            val_mask = np.isin(y_val, self.train_classes)
            X_val_filtered = X_val[val_mask]
            y_val_filtered = y_val[val_mask]

            if verbose:
                print(f"      - Original validation samples: {len(X_val)}")
                print(f"      - Filtered validation samples: {len(X_val_filtered)}")

            if len(X_val_filtered) == 0:
                if verbose:
                    print(f"      ⚠️ No valid validation samples! Returning zeros.")
                return 0.0, 0.0, None

            # Encode validation labels
            y_val_encoded = self.label_encoder.transform(y_val_filtered)

            # Predict
            y_pred = self.predict(X_val_filtered)

            # Calculate metrics
            accuracy = accuracy_score(y_val_encoded, y_pred)
            f1 = f1_score(y_val_encoded, y_pred, average='weighted', zero_division=0)

            if verbose:
                print(f"      ✅ Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

            return accuracy, f1, self.base_model

        elif self.handle_unknown == 'ignore':
            # Create mask for known classes
            known_mask = np.isin(y_val, self.train_classes)

            if not np.any(known_mask):
                if verbose:
                    print(f"      ⚠️ No known classes in validation!")
                return 0.0, 0.0, None

            # Process known classes
            X_known = X_val[known_mask]
            y_known = y_val[known_mask]
            y_known_encoded = self.label_encoder.transform(y_known)

            # Predict on all data but only evaluate on known classes
            y_pred_all = self.predict(X_val)
            y_pred_known = y_pred_all[known_mask]

            # Calculate metrics only on known classes
            accuracy = accuracy_score(y_known_encoded, y_pred_known)
            f1 = f1_score(y_known_encoded, y_pred_known, average='weighted', zero_division=0)

            if verbose:
                print(f"      - Known samples: {np.sum(known_mask)}/{len(X_val)}")
                print(f"      ✅ Accuracy (known only): {accuracy:.4f}, F1: {f1:.4f}")

            return accuracy, f1, self.base_model


# ============================================================
# SOLUTION 2: Preprocessing function for class alignment
# ============================================================

def align_classes_for_evaluation(y_train, y_val, X_val):
    """
    Align validation classes with training classes
    Returns filtered X_val and y_val, plus statistics
    """
    train_classes = set(np.unique(y_train))
    val_classes = set(np.unique(y_val))

    # Find common and unknown classes
    common_classes = train_classes.intersection(val_classes)
    unknown_classes = val_classes - train_classes

    # Filter validation data
    val_mask = np.isin(y_val, list(train_classes))
    X_val_filtered = X_val[val_mask]
    y_val_filtered = y_val[val_mask]

    stats = {
        'train_classes': len(train_classes),
        'val_classes': len(val_classes),
        'common_classes': len(common_classes),
        'unknown_classes': len(unknown_classes),
        'original_samples': len(X_val),
        'filtered_samples': len(X_val_filtered)
    }

    return X_val_filtered, y_val_filtered, stats


# ============================================================
# SOLUTION 3: Evaluation function with multiple strategies
# ============================================================

def evaluate_model_robust(model, X_train, y_train, X_val, y_val, model_name, strategy='auto'):
    """
    Robust evaluation function with multiple strategies for handling class mismatch

    Strategies:
    - 'auto': Automatically choose best strategy
    - 'filter': Filter out unknown classes
    - 'ignore': Keep all but only evaluate on known classes
    - 'report': Report the mismatch and continue
    """

    print(f"\n📊 Training {model_name}...")
    print(f"  - Training classes: {len(np.unique(y_train))}")
    print(f"  - Validation classes: {len(np.unique(y_val))}")
    print(f"  - Training samples: {len(X_train)}")
    print(f"  - Validation samples: {len(X_val)}")

    # Check for class mismatch
    train_classes = set(np.unique(y_train))
    val_classes = set(np.unique(y_val))

    missing_in_val = train_classes - val_classes
    new_in_val = val_classes - train_classes

    if missing_in_val or new_in_val:
        print(f"  ⚠️ Class mismatch detected:")
        if missing_in_val:
            print(f"    - Classes in train but not in val: {len(missing_in_val)}")
        if new_in_val:
            print(f"    - Classes in val but not in train: {len(new_in_val)}")

    # Apply strategy
    if strategy == 'filter' or (strategy == 'auto' and new_in_val):
        # Filter validation data
        val_mask = np.isin(y_val, list(train_classes))
        X_val_filtered = X_val[val_mask]
        y_val_filtered = y_val[val_mask]

        print(f"  - Validation samples after filtering: {len(X_val_filtered)}")

        if len(X_val_filtered) == 0:
            print(f"  ❌ No validation samples left after filtering!")
            return 0, 0, None

        y_train_to_use = y_train
        y_val_to_use = y_val_filtered
        X_val_to_use = X_val_filtered

    elif strategy == 'ignore' or strategy == 'auto':
        # Keep all but we'll handle during evaluation
        # Need to encode labels first
        le = LabelEncoder()
        le.fit(y_train)

        y_train_encoded = le.transform(y_train)

        # For validation, mark unknown classes
        y_val_encoded = np.array([
            le.transform([y])[0] if y in le.classes_ else -1
            for y in y_val
        ])

        # Only evaluate on known classes
        known_mask = y_val_encoded != -1

        if not np.any(known_mask):
            print(f"  ❌ No known classes in validation!")
            return 0, 0, None

        y_train_to_use = y_train_encoded
        y_val_to_use = y_val_encoded[known_mask]
        X_val_to_use = X_val[known_mask]

        print(f"  - Known validation samples: {np.sum(known_mask)}/{len(X_val)}")

    else:  # 'report' strategy - train anyway but expect errors
        y_train_to_use = y_train
        y_val_to_use = y_val
        X_val_to_use = X_val

    # Train the model
    try:
        model.fit(X_train, y_train_to_use)

        # Predict
        y_pred = model.predict(X_val_to_use)

        # Calculate metrics
        accuracy = accuracy_score(y_val_to_use, y_pred)
        f1 = f1_score(y_val_to_use, y_pred, average='weighted', zero_division=0)

        print(f"  ✅ Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

        return accuracy, f1, model

    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        return 0, 0, None


# ============================================================
# MAIN EXECUTION CELL
# ============================================================

print("="*70)
print("TIME SERIES CLASSIFICATION WITH ROBUST CLASS HANDLING")
print("="*70)

# Define base models
base_models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    ),
    'CatBoost': CatBoostClassifier(
        iterations=100,
        depth=6,
        learning_rate=0.1,
        random_seed=42,
        verbose=False
    )
}

# Define feature sets (replace with your actual data)
feature_sets = [
    ('ARIMA', X_train_arima, X_val_arima),
    ('Prophet', X_train_prophet, X_val_prophet),
    ('Hybrid', X_train_hybrid, X_val_hybrid)
]

# ============================================================
# APPROACH 1: Using RobustTimeSeriesClassifier wrapper
# ============================================================

print("\n" + "="*70)
print("APPROACH 1: RobustTimeSeriesClassifier")
print("="*70)

results_robust = {}
trained_models_robust = {}

for feat_name, X_tr, X_va in feature_sets:
    print(f"\n{'-'*50}")
    print(f"FEATURE SET: {feat_name}")
    print(f"{'-'*50}")

    for model_name, base_model in base_models.items():
        full_name = f"{feat_name}_{model_name}"

        # Create robust classifier
        robust_model = RobustTimeSeriesClassifier(
            base_model=base_model.__class__(**base_model.get_params()),
            model_name=full_name,
            handle_unknown='filter'  # Can be 'filter', 'ignore', or 'map'
        )

        # Fit the model
        robust_model.fit(X_tr, y_train)

        # Evaluate
        accuracy, f1, trained = robust_model.evaluate(X_va, y_val, verbose=True)

        results_robust[full_name] = (accuracy, f1)
        if trained is not None:
            trained_models_robust[full_name] = robust_model

# ============================================================
# APPROACH 2: Using preprocessing alignment
# ============================================================

print("\n" + "="*70)
print("APPROACH 2: Preprocessing Alignment")
print("="*70)

results_aligned = {}
trained_models_aligned = {}

for feat_name, X_tr, X_va in feature_sets:
    print(f"\n{'-'*50}")
    print(f"FEATURE SET: {feat_name}")
    print(f"{'-'*50}")

    # Align classes first
    X_val_aligned, y_val_aligned, stats = align_classes_for_evaluation(y_train, y_val, X_va)

    print(f"  Alignment stats:")
    print(f"    - Training classes: {stats['train_classes']}")
    print(f"    - Validation classes: {stats['val_classes']}")
    print(f"    - Common classes: {stats['common_classes']}")
    print(f"    - Unknown classes: {stats['unknown_classes']}")
    print(f"    - Samples kept: {stats['filtered_samples']}/{stats['original_samples']}")

    if stats['filtered_samples'] == 0:
        print(f"  ⚠️ No aligned samples! Skipping...")
        continue

    for model_name, base_model in base_models.items():
        full_name = f"{feat_name}_{model_name}"

        # Create fresh instance
        new_model = base_model.__class__(**base_model.get_params())

        try:
            # Train on all training data
            new_model.fit(X_tr, y_train)

            # Predict on aligned validation
            y_pred = new_model.predict(X_val_aligned)

            # Calculate metrics
            accuracy = accuracy_score(y_val_aligned, y_pred)
            f1 = f1_score(y_val_aligned, y_pred, average='weighted', zero_division=0)

            results_aligned[full_name] = (accuracy, f1)
            trained_models_aligned[full_name] = new_model

            print(f"  ✅ {full_name}: Acc={accuracy:.4f}, F1={f1:.4f}")

        except Exception as e:
            print(f"  ❌ {full_name}: {str(e)[:100]}")

# ============================================================
# APPROACH 3: Using the robust evaluation function
# ============================================================

print("\n" + "="*70)
print("APPROACH 3: Robust Evaluation Function")
print("="*70)

results_eval = {}
trained_models_eval = {}

for feat_name, X_tr, X_va in feature_sets:
    print(f"\n{'-'*50}")
    print(f"FEATURE SET: {feat_name}")
    print(f"{'-'*50}")

    for model_name, base_model in base_models.items():
        full_name = f"{feat_name}_{model_name}"

        # Create fresh instance
        new_model = base_model.__class__(**base_model.get_params())

        # Use robust evaluation with auto strategy
        accuracy, f1, trained = evaluate_model_robust(
            new_model, X_tr, y_train, X_va, y_val,
            full_name, strategy='auto'
        )

        results_eval[full_name] = (accuracy, f1)
        if trained is not None:
            trained_models_eval[full_name] = trained

# ============================================================
# COMPARE ALL APPROACHES
# ============================================================

print("\n" + "="*70)
print("MODEL COMPARISON - ALL APPROACHES")
print("="*70)

# Combine results from all approaches
all_results = {}

for name, acc, f1 in [(n, *results_robust[n]) for n in results_robust]:
    all_results[f"{name}_Robust"] = (acc, f1)

for name, acc, f1 in [(n, *results_aligned[n]) for n in results_aligned]:
    all_results[f"{name}_Aligned"] = (acc, f1)

for name, acc, f1 in [(n, *results_eval[n]) for n in results_eval]:
    all_results[f"{name}_Eval"] = (acc, f1)

# Convert to DataFrame
comparison_df = pd.DataFrame(all_results, index=['Accuracy', 'F1']).T
comparison_df = comparison_df.sort_values('F1', ascending=False)

print("\nTop 10 Models by F1 Score:")
print(comparison_df.head(10).to_string())

# Find best model overall
best_model_name = comparison_df.index[0]
best_f1 = comparison_df.loc[best_model_name, 'F1']
best_acc = comparison_df.loc[best_model_name, 'Accuracy']

print(f"\n{'🏆'*10} BEST MODEL {'🏆'*10}")
print(f"Model: {best_model_name}")
print(f"F1 Score: {best_f1:.4f}")
print(f"Accuracy: {best_acc:.4f}")

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

summary = {
    'Approach': ['Robust Classifier', 'Alignment', 'Eval Function'],
    'Models Evaluated': [len(results_robust), len(results_aligned), len(results_eval)],
    'Avg F1': [
        np.mean([f1 for _, f1 in results_robust.values()]),
        np.mean([f1 for _, f1 in results_aligned.values()]),
        np.mean([f1 for _, f1 in results_eval.values()])
    ],
    'Max F1': [
        np.max([f1 for _, f1 in results_robust.values()]),
        np.max([f1 for _, f1 in results_aligned.values()]),
        np.max([f1 for _, f1 in results_eval.values()])
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# ============================================================
# SAVE RESULTS
# ============================================================

# Save detailed comparison
comparison_df.to_csv("model_comparison_all_approaches.csv")
print(f"\n✅ Detailed results saved to 'model_comparison_all_approaches.csv'")

# Save best model
print(f"\n✅ Best model: {best_model_name}")
print(f"   F1 Score: {best_f1:.4f}")
print(f"   Accuracy: {best_acc:.4f}")

# Optional: Save the best model
if best_model_name.endswith('_Robust'):
    best_model = trained_models_robust[best_model_name.replace('_Robust', '')]
elif best_model_name.endswith('_Aligned'):
    best_model = trained_models_aligned[best_model_name.replace('_Aligned', '')]
elif best_model_name.endswith('_Eval'):
    best_model = trained_models_eval[best_model_name.replace('_Eval', '')]

# You can save the best model using joblib
# import joblib
# joblib.dump(best_model, 'best_model.pkl')
# print("✅ Best model saved to 'best_model.pkl'")

TIME SERIES CLASSIFICATION WITH ROBUST CLASS HANDLING

APPROACH 1: RobustTimeSeriesClassifier

--------------------------------------------------
FEATURE SET: ARIMA
--------------------------------------------------
    Fitting ARIMA_XGBoost...
      - Training classes: 184
    Evaluating ARIMA_XGBoost...
      - Training classes: 184
      - Validation classes: 94
      - Unknown classes: 40
      - Original validation samples: 1020
      - Filtered validation samples: 912
      ✅ Accuracy: 0.9002, F1: 0.8822
    Fitting ARIMA_LightGBM...
      - Training classes: 184
    Evaluating ARIMA_LightGBM...
      - Training classes: 184
      - Validation classes: 94
      - Unknown classes: 40
      - Original validation samples: 1020
      - Filtered validation samples: 912
      ✅ Accuracy: 0.1513, F1: 0.0398
    Fitting ARIMA_CatBoost...
      - Training classes: 184
    Evaluating ARIMA_CatBoost...
      - Training classes: 184
      - Validation classes: 94
      - Unknown classes: 40


**Cell 14: Retrain best model on full training+validation data and evaluate on test**

In [15]:
# ============================================================
# Cell 14 (FIXED): Retrain best model on train+val and evaluate on test
# ============================================================

# Determine which feature set corresponds to the best model
if 'ARIMA' in best_model_name:
    X_full = pd.concat([X_train_arima, X_val_arima])
    X_test_best = X_test_arima
    best_feat_type = 'arima'
elif 'Prophet' in best_model_name:
    X_full = pd.concat([X_train_prophet, X_val_prophet])
    X_test_best = X_test_prophet
    best_feat_type = 'prophet'
else:
    X_full = pd.concat([X_train_hybrid, X_val_hybrid])
    X_test_best = X_test_hybrid
    best_feat_type = 'hybrid'

y_full = pd.concat([y_train, y_val])

# Recreate best model with same hyperparameters
if 'XGBoost' in best_model_name:
    final_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
elif 'LightGBM' in best_model_name:
    final_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )
else:
    final_model = CatBoostClassifier(
        iterations=100,
        depth=6,
        learning_rate=0.1,
        random_seed=42,
        verbose=False
    )

print(f"\n🔄 Retraining {best_model_name} on combined train+val ({len(X_full)} samples)...")
final_model.fit(X_full, y_full)

# --- Test set evaluation with safety checks ---
print("\n📊 Evaluating on test set...")

if X_test_best.shape[0] == 0 or y_test.shape[0] == 0:
    print("⚠️ Test set is empty. Cannot compute test metrics.")
    test_acc = np.nan
    test_f1 = np.nan
else:
    # Make predictions
    test_pred = final_model.predict(X_test_best)

    # Ensure predictions are of the same type as y_test
    if test_pred.dtype != y_test.dtype:
        test_pred = test_pred.astype(y_test.dtype)

    # Check that there is at least one valid sample
    if len(test_pred) == 0:
        print("⚠️ No predictions generated.")
        test_acc = np.nan
        test_f1 = np.nan
    else:
        # Calculate accuracy and weighted F1 (handle zero division)
        test_acc = accuracy_score(y_test, test_pred)
        test_f1 = f1_score(y_test, test_pred, average='weighted', zero_division=0)

print(f"\n✅ Final Model Test Performance:")
print(f"   Test Accuracy: {test_acc:.6f}")
print(f"   Test Weighted F1: {test_f1:.6f}")

# If test set was empty, optionally evaluate on validation set for reference
if pd.isna(test_acc) and X_val_arima.shape[0] > 0:
    print("\n🔄 Falling back to validation set evaluation (for reference only)...")
    # Use the appropriate validation set based on best feature type
    if best_feat_type == 'arima':
        X_val_best = X_val_arima
    elif best_feat_type == 'prophet':
        X_val_best = X_val_prophet
    else:
        X_val_best = X_val_hybrid

    val_pred = final_model.predict(X_val_best)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred, average='weighted', zero_division=0)
    print(f"   Validation Accuracy: {val_acc:.6f}")
    print(f"   Validation Weighted F1: {val_f1:.6f}")


🔄 Retraining Prophet_XGBoost_Robust on combined train+val (3688 samples)...

📊 Evaluating on test set...
⚠️ Test set is empty. Cannot compute test metrics.

✅ Final Model Test Performance:
   Test Accuracy: nan
   Test Weighted F1: nan

🔄 Falling back to validation set evaluation (for reference only)...
   Validation Accuracy: 0.989216
   Validation Weighted F1: 0.985889


**Cell 15: Save all artifacts for later use**

In [39]:
import pickle
import pandas as pd

# --- Safe retrieval of feature columns ---
if X_full is not None and hasattr(X_full, 'columns'):
    feature_columns = X_full.columns.tolist()
    print(f"✅ Feature columns retrieved from X_full: {len(feature_columns)} features")
else:
    print("⚠️  X_full is None or not a DataFrame. Attempting to get feature names from model...")
    if hasattr(final_model, 'feature_names_in_'):
        feature_columns = final_model.feature_names_in_.tolist()
        print(f"✅ Feature names retrieved from model: {len(feature_columns)} features")
    else:
        feature_columns = []
        print("⚠️  Could not retrieve feature names. 'feature_columns' will be empty.")

# Build artifacts dictionary
artifacts = {
    'final_model': final_model,
    'best_model_name': best_model_name,
    'best_feature_type': best_feat_type,
    'label_encoders': label_encoders,
    'scaler': scaler,
    'course_uni_mapping': course_uni_mapping,
    'aptitude_courses_set': list(aptitude_courses_set),
    'stream_course_names': {k: list(v) for k, v in stream_course_names.items()},
    'cutoff_lookup': cutoff_lookup,
    'feature_columns': feature_columns   # now safely defined
}

# Save the main artifacts file
with open('ugc_final_recommendation_system.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

# Also save individual components
with open('final_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
with open('label_encoders_final.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open('scaler_final.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\n✅ All artifacts saved successfully.")
print("📁 Files: ugc_final_recommendation_system.pkl, final_model.pkl, label_encoders_final.pkl, scaler_final.pkl")

⚠️  X_full is None or not a DataFrame. Attempting to get feature names from model...
⚠️  Could not retrieve feature names. 'feature_columns' will be empty.

✅ All artifacts saved successfully.
📁 Files: ugc_final_recommendation_system.pkl, final_model.pkl, label_encoders_final.pkl, scaler_final.pkl


In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Cell 16: Define the recommendation function (with cut‑off filter and stream eligibility)**

In [40]:
# Cell 16: Recommendation function using trained model
def recommend_courses(user_input, model, feature_type, top_n=10, use_cutoff=True):
    """
    Generate top‑N course recommendations.
    - user_input: dict with all required fields (Stream, Z_Score, Island_Rank, etc.)
    - model: trained classifier
    - feature_type: 'arima', 'prophet', or 'hybrid'
    - top_n: number of recommendations
    - use_cutoff: if True, filter courses where student's Z‑score is below cut‑off mean
    """
    stream = user_input.get('Stream', 'Unknown')
    if stream not in stream_course_names:
        print(f"⚠️ Unknown stream '{stream}'. No recommendations.")
        return {"recommendations": [], "input_summary": {}}

    # Prepare input dataframe
    input_df = pd.DataFrame([user_input])

    # Encode categorical features using stored encoders
    for col, le in label_encoders.items():
        if col in input_df.columns:
            val = str(input_df[col].iloc[0])
            if val in le.classes_:
                input_df[col] = le.transform([val])[0]
            else:
                input_df[col] = 0  # fallback

    # Add grade numeric features
    grade_map = {"A":4, "B":3, "C":2, "S":1, "F":0}
    for g in ["Grade_1","Grade_2","Grade_3"]:
        if g in input_df.columns:
            input_df[f"{g}_num"] = grade_map.get(input_df[g].iloc[0], 1)
        else:
            input_df[f"{g}_num"] = 1
    input_df["Avg_Grade"] = (input_df.get("Grade_1_num",1) + input_df.get("Grade_2_num",1) + input_df.get("Grade_3_num",1))/3

    # Interest questions (default 3)
    for col in interest_cols:
        if col not in input_df.columns:
            input_df[col] = 3
    input_df["STEM_Interest"] = (input_df["q1_science_tech"] + input_df["q4_data"] + input_df["q9_innovation"])/3
    input_df["Healthcare_Interest"] = (input_df["q2_healthcare"] + input_df["q10_people_social"])/2
    input_df["Creative_Interest"] = (input_df["q3_design"] + input_df["q6_arts_culture"])/2
    input_df["Nature_Interest"] = (input_df["q7_nature_env"] + input_df["q8_hands_on"])/2
    input_df["Business_Interest"] = (input_df["q5_business"] + input_df["q11_urban_corporate"] + input_df["q12_flexible_path"])/3

    # For forecast features, we need to set Z_Score appropriately
    # In production you would compute actual forecasts, here we use original Z_Score as placeholder
    input_df['Z_Score'] = user_input.get('Z_Score', 0)
    # Also set ARIMA_Z, Simple_Forecast_Z, Hybrid_Z to the same value (or compute them)
    # For simplicity, copy Z_Score to these fields; they will be overwritten later if needed
    input_df['ARIMA_Z'] = input_df['Z_Score']
    input_df['Simple_Forecast_Z'] = input_df['Z_Score']
    input_df['Hybrid_Z'] = input_df['Z_Score']

    # Ensure all feature columns from training are present
    train_cols = X_full.columns.tolist()
    for col in train_cols:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[train_cols]

    # Scale numerical columns
    input_df_scaled = scaler.transform(input_df)
    input_df = pd.DataFrame(input_df_scaled, columns=train_cols)

    # Get probabilities
    try:
        proba = model.predict_proba(input_df)[0]
    except Exception as e:
        print(f"❌ Prediction error: {e}")
        return {"recommendations": [], "input_summary": {}}

    # Eligible courses for this stream
    eligible_course_names = stream_course_names[stream]
    eligible_indices = []
    for idx, course_name in enumerate(label_encoders['Course'].classes_):
        if course_name in eligible_course_names:
            eligible_indices.append(idx)

    if not eligible_indices:
        print(f"⚠️ No eligible courses found for stream '{stream}'.")
        return {"recommendations": [], "input_summary": {}}

    # Build list of (idx, score) for eligible courses
    course_scores = [(idx, proba[idx]) for idx in eligible_indices]
    course_scores.sort(key=lambda x: x[1], reverse=True)

    z_student = user_input.get('Z_Score', 0)
    recommendations = []
    used_univ = set()

    for idx, score in course_scores:
        if len(recommendations) >= top_n:
            break
        course_name = label_encoders['Course'].inverse_transform([idx])[0]
        # Find university info from mapping
        uni_info = None
        course_upper = course_name.upper()
        for mapped_course, info in course_uni_mapping.items():
            if mapped_course in course_upper or course_upper in mapped_course:
                uni_info = info
                break
        if uni_info is None:
            continue
        university = uni_info['University']
        uni_code = uni_info['Uni_Code']
        aptitude = uni_info['Aptitude']

        # Cut‑off filter
        if use_cutoff:
            key = (course_upper, university.upper())
            if key in cutoff_lookup:
                cutoff_mean = cutoff_lookup[key]['mean']
                if z_student < cutoff_mean * 0.95:   # allow 5% margin
                    continue
            # if no cut‑off data, allow

        # Diversity: at most 2 courses from the same university
        if university in used_univ and len([r for r in recommendations if r['university']==university]) >= 2:
            continue
        used_univ.add(university)

        recommendations.append({
            'rank': len(recommendations)+1,
            'course': course_name,
            'score': round(score, 4),
            'university': university,
            'uni_code': uni_code,
            'aptitude_required': aptitude
        })

    input_summary = {
        'stream': stream,
        'z_score': user_input.get('Z_Score', 0),
        'island_rank': user_input.get('Island_Rank', 0),
        'district': user_input.get('District', 'Unknown')
    }
    return {'recommendations': recommendations, 'input_summary': input_summary}

**Cell 17: Test the recommendation function with sample inputs**

In [41]:
# ============================================================
# FINAL RECOMMENDATION SYSTEM – UGC COURSE PREDICTOR
# ============================================================

import json
import pickle
import pandas as pd
import numpy as np
import os
import random
from tabulate import tabulate

# ------------------------------------------------------------
# 1. CORRECTED STREAM ELIGIBILITY MAP (from your provided data)
# ------------------------------------------------------------
STREAM_ELIGIBILITY = {
    "Arts": [
        ("Arts", "019A", "University of Colombo", "No"),
        ("Arts", "019B", "University of Peradeniya", "No"),
        ("Arts", "019C", "University of Sri Jayewardenepura", "No"),
        ("Arts", "019D", "University of Kelaniya", "No"),
        ("Arts", "019E", "University of Jaffna", "No"),
        ("Arts", "019F", "University of Ruhuna", "No"),
        ("Arts", "019H", "Eastern University, Sri Lanka", "No"),
        ("Arts", "019J", "South Eastern University of Sri Lanka", "No"),
        ("Arts", "019K", "Rajarata University of Sri Lanka", "No"),
        ("Arts (SP) - Mass Media", "020S", "Sripalee Campus, University of Colombo", "Yes"),
        ("Arts (SP) - Performing Arts", "041S", "Sripalee Campus, University of Colombo", "Yes"),
        ("Arts (SAB)", "021L", "Sabaragamuwa University of Sri Lanka", "No"),
        ("Communication Studies", "029W", "Trincomalee Campus, Eastern University", "No"),
        ("Peace & Conflict Resolution", "031D", "University of Kelaniya", "No"),
        ("Islamic Studies", "063J", "South Eastern University of Sri Lanka", "No"),
        ("Arabic Language", "084J", "South Eastern University of Sri Lanka", "No"),
        ("Teaching English as a Second Language (TESL)", "105C", "University of Sri Jayewardenepura", "No"),
        ("Teaching English as a Second Language (TESL)", "105D", "University of Kelaniya", "No"),
        ("Teaching English as a Second Language (TESL)", "105L", "Sabaragamuwa University of Sri Lanka", "No"),
        ("Social Work", "112B", "University of Peradeniya", "No"),
        ("Social Work", "112C", "University of Sri Jayewardenepura", "No"),
        ("Arts - Information Technology", "128C", "University of Sri Jayewardenepura", "No")
    ],
    "Commerce": [
        ("Management", "016A", "University of Colombo", "No"),
        ("Management", "016B", "University of Peradeniya", "No"),
        ("Management", "016C", "University of Sri Jayewardenepura", "No"),
        ("Management", "016D", "University of Kelaniya", "No"),
        ("Management", "016E", "University of Jaffna", "No"),
        ("Management", "016F", "University of Ruhuna", "No"),
        ("Management", "016H", "Eastern University", "No"),
        ("Management", "016J", "South Eastern University", "No"),
        ("Management", "016K", "Rajarata University", "No"),
        ("Management", "016L", "Sabaragamuwa University", "No"),
        ("Management", "016M", "Wayamba University", "No"),
        ("Management and Public Policy", "028C", "University of Sri Jayewardenepura", "No"),
        ("Real Estate Management and Valuation", "017C", "University of Sri Jayewardenepura", "No"),
        ("Commerce", "018C", "University of Sri Jayewardenepura", "No"),
        ("Commerce", "018D", "University of Kelaniya", "No"),
        ("Commerce", "018E", "University of Jaffna", "No"),
        ("Commerce", "018H", "Eastern University", "No"),
        ("Commerce", "018J", "South Eastern University", "No"),
        ("Management Studies (TV)", "022W", "Trincomalee Campus", "No"),
        ("Management Studies (TV)", "022R", "University of Vavuniya", "No"),
        ("Business Information Systems (Honours) (BIS)", "077C", "University of Sri Jayewardenepura", "No"),
        ("Accounting Information Systems", "127D", "University of Kelaniya", "No"),
        ("Banking and Insurance", "133R", "University of Vavuniya", "No"),
        ("Service Management", "140P", "Gampaha Wickramarachchi University", "No")
    ],
    "Biological Science": [
        ("Medicine", "001A", "University of Colombo", "No"),
        ("Medicine", "001B", "University of Peradeniya", "No"),
        ("Medicine", "001C", "University of Sri Jayewardenepura", "No"),
        ("Medicine", "001D", "University of Kelaniya", "No"),
        ("Medicine", "001E", "University of Jaffna", "No"),
        ("Medicine", "001F", "University of Ruhuna", "No"),
        ("Medicine", "001G", "University of Moratuwa", "No"),
        ("Medicine", "001H", "Eastern University", "No"),
        ("Medicine", "001K", "Rajarata University", "No"),
        ("Medicine", "001L", "Sabaragamuwa University", "No"),
        ("Medicine", "001M", "Wayamba University", "No"),
        ("Medicine", "001U", "Uva Wellassa University", "No"),
        ("Dental Surgery", "002B", "University of Peradeniya", "No"),
        ("Dental Surgery", "002C", "University of Sri Jayewardenepura", "No"),
        ("Veterinary Science", "003B", "University of Peradeniya", "No"),
        ("Agriculture", "004E", "University of Jaffna", "No"),
        ("Agriculture", "004H", "Eastern University", "No"),
        ("Agriculture", "004K", "Rajarata University", "No"),
        ("Agriculture", "004L", "Sabaragamuwa University", "No"),
        ("Agriculture", "004M", "Wayamba University", "No"),
        ("Food Science & Nutrition", "005M", "Wayamba University", "No"),
        ("Biological Science", "006A", "University of Colombo", "No"),
        ("Biological Science", "006B", "University of Peradeniya", "No"),
        ("Biological Science", "006C", "University of Sri Jayewardenepura", "No"),
        ("Biological Science", "006D", "University of Kelaniya", "No"),
        ("Biological Science", "006E", "University of Jaffna", "No"),
        ("Biological Science", "006F", "University of Ruhuna", "No"),
        ("Biological Science", "006H", "Eastern University", "No"),
        ("Biological Science", "006J", "South Eastern University", "No"),
        ("Applied Sciences (Biological Sc.)", "007K", "Rajarata University", "No"),
        ("Applied Sciences (Biological Sc.)", "007L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Biological Sc.)", "007R", "University of Vavuniya", "No"),
        ("Ayurveda Medicine and Surgery", "032A", "University of Colombo", "No"),
        ("Ayurveda Medicine and Surgery", "032P", "Gampaha Wickramarachchi University", "No"),
        ("Unani Medicine and Surgery", "033A", "University of Colombo", "No"),
        ("Siddha Medicine and Surgery", "036E", "University of Jaffna", "No"),
        ("Siddha Medicine and Surgery", "036W", "Trincomalee Campus", "No"),
        ("Nursing", "037A", "University of Colombo", "No"),
        ("Nursing", "037B", "University of Peradeniya", "No"),
        ("Nursing", "037C", "University of Sri Jayewardenepura", "No"),
        ("Nursing", "037E", "University of Jaffna", "No"),
        ("Nursing", "037F", "University of Ruhuna", "No"),
        ("Nursing", "037H", "Eastern University", "No"),
        ("Pharmacy", "051B", "University of Peradeniya", "No"),
        ("Pharmacy", "051C", "University of Sri Jayewardenepura", "No"),
        ("Pharmacy", "051E", "University of Jaffna", "No"),
        ("Pharmacy", "051F", "University of Ruhuna", "No"),
        ("Medical Laboratory Sciences", "052B", "University of Peradeniya", "No"),
        ("Medical Laboratory Sciences", "052C", "University of Sri Jayewardenepura", "No"),
        ("Medical Laboratory Sciences", "052E", "University of Jaffna", "No"),
        ("Medical Laboratory Sciences", "052F", "University of Ruhuna", "No"),
        ("Radiography", "053B", "University of Peradeniya", "No"),
        ("Physiotherapy", "054A", "University of Colombo", "No"),
        ("Physiotherapy", "054B", "University of Peradeniya", "No"),
        ("Health Promotion", "050K", "Rajarata University", "No")
    ],
    "Physical Science": [
        ("Engineering", "008B", "University of Peradeniya", "No"),
        ("Engineering", "008C", "University of Sri Jayewardenepura", "No"),
        ("Engineering", "008E", "University of Jaffna", "No"),
        ("Engineering", "008F", "University of Ruhuna", "No"),
        ("Engineering", "008G", "University of Moratuwa", "No"),
        ("Engineering", "008J", "South Eastern University", "No"),
        ("Engineering (EM)", "009G", "University of Moratuwa", "No"),
        ("Engineering (TM)", "010G", "University of Moratuwa", "No"),
        ("Quantity Surveying", "011G", "University of Moratuwa", "No"),
        ("Computer Science", "012C", "University of Sri Jayewardenepura", "No"),
        ("Computer Science", "012D", "University of Kelaniya", "No"),
        ("Computer Science", "012E", "University of Jaffna", "No"),
        ("Computer Science", "012F", "University of Ruhuna", "No"),
        ("Computer Science", "012T", "University of Colombo School of Computing", "No"),
        ("Computer Science", "012W", "Trincomalee Campus", "No"),
        ("Physical Science", "013A", "University of Colombo", "No"),
        ("Physical Science", "013B", "University of Peradeniya", "No"),
        ("Physical Science", "013C", "University of Sri Jayewardenepura", "No"),
        ("Physical Science", "013D", "University of Kelaniya", "No"),
        ("Physical Science", "013E", "University of Jaffna", "No"),
        ("Physical Science", "013F", "University of Ruhuna", "No"),
        ("Physical Science", "013H", "Eastern University", "No"),
        ("Physical Science", "013J", "South Eastern University", "No"),
        ("Surveying Science", "014L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Physical Sc.)", "015K", "Rajarata University", "No"),
        ("Applied Sciences (Physical Sc.)", "015L", "Sabaragamuwa University", "No"),
        ("Applied Sciences (Physical Sc.)", "015M", "Wayamba University", "No"),
        ("Applied Sciences (Physical Sc.)", "015R", "University of Vavuniya", "No"),
        ("Applied Sciences (Physical Sc.)", "015W", "Trincomalee Campus", "No")
    ],
    "Engineering Technology": [
        ("Engineering Technology (ET)", "102A", "University of Colombo", "No"),
        ("Engineering Technology (ET)", "102C", "University of Sri Jayewardenepura", "No"),
        ("Engineering Technology (ET)", "102D", "University of Kelaniya", "No"),
        ("Engineering Technology (ET)", "102E", "University of Jaffna", "No"),
        ("Engineering Technology (ET)", "102F", "University of Ruhuna", "No"),
        ("Engineering Technology (ET)", "102K", "Rajarata University", "No"),
        ("Engineering Technology (ET)", "102L", "Sabaragamuwa University", "No"),
        ("Engineering Technology (ET)", "102M", "Wayamba University", "No"),
        ("Engineering Technology (ET)", "102U", "Uva Wellassa University", "No")
    ],
    "Biosystems Technology": [
        ("Biosystems Technology (BST)", "103A", "University of Colombo", "No"),
        ("Biosystems Technology (BST)", "103C", "University of Sri Jayewardenepura", "No"),
        ("Biosystems Technology (BST)", "103D", "University of Kelaniya", "No"),
        ("Biosystems Technology (BST)", "103E", "University of Jaffna", "No"),
        ("Biosystems Technology (BST)", "103F", "University of Ruhuna", "No"),
        ("Biosystems Technology (BST)", "103H", "Eastern University", "No"),
        ("Biosystems Technology (BST)", "103J", "South Eastern University", "No"),
        ("Biosystems Technology (BST)", "103K", "Rajarata University", "No"),
        ("Biosystems Technology (BST)", "103L", "Sabaragamuwa University", "No"),
        ("Biosystems Technology (BST)", "103M", "Wayamba University", "No"),
        ("Biosystems Technology (BST)", "103U", "Uva Wellassa University", "No")
    ],
    "Common / Multi-Stream": [
        ("Information Technology (IT)", "026G", "University of Moratuwa", "No"),
        ("Management and Information Technology (MIT)", "027D", "University of Kelaniya", "No"),
        ("Quantity Surveying", "011G", "University of Moratuwa", "No"),
        ("Surveying Science", "014L", "Sabaragamuwa University", "No"),
        ("Urban Informatics and Planning", "030G", "University of Moratuwa", "No"),
        ("Architecture", "023G", "University of Moratuwa", "Yes"),
        ("Fashion Design & Product Development", "034G", "University of Moratuwa", "Yes"),
        ("Landscape Architecture", "097G", "University of Moratuwa", "Yes"),
        ("Design", "024G", "University of Moratuwa", "Yes"),
        ("Law", "025A", "University of Colombo", "Yes"),
        ("Law", "025B", "University of Peradeniya", "Yes"),
        ("Law", "025E", "University of Jaffna", "Yes"),
        ("Facilities Management", "056G", "University of Moratuwa", "No"),
        ("Management and Information Technology (SEUSL)", "079J", "South Eastern University", "No"),
        ("Science and Technology", "064U", "Uva Wellassa University", "No"),
        ("Computer Science & Technology", "065U", "Uva Wellassa University", "No"),
        ("Entrepreneurship and Management", "066U", "Uva Wellassa University", "No"),
        ("Industrial Information Technology", "075U", "Uva Wellassa University", "No"),
        ("Mineral Resources and Technology", "076U", "Uva Wellassa University", "No"),
        ("Hospitality, Tourism and Events Management", "090U", "Uva Wellassa University", "No"),
        ("Physical Education", "081E", "University of Jaffna", "Yes"),
        ("Physical Education", "081L", "Sabaragamuwa University", "Yes"),
        ("Sports Science & Management", "082C", "University of Sri Jayewardenepura", "Yes"),
        ("Sports Science & Management", "082D", "University of Kelaniya", "Yes"),
        ("Sports Science & Management", "082L", "Sabaragamuwa University", "Yes"),
        ("Information Technology & Management", "091G", "University of Moratuwa", "No"),
        ("Tourism & Hospitality Management", "092K", "Rajarata University", "No"),
        ("Tourism & Hospitality Management", "092L", "Sabaragamuwa University", "No"),
        ("Agricultural Resource Management and Technology", "093F", "University of Ruhuna", "No"),
        ("Agribusiness Management", "094F", "University of Ruhuna", "No"),
        ("Green Technology", "095F", "University of Ruhuna", "No"),
        ("Information Systems", "096C", "University of Sri Jayewardenepura", "No"),
        ("Information Systems", "096L", "Sabaragamuwa University", "No"),
        ("Information Systems", "096T", "UCSC", "No"),
        ("Translation Studies", "098D", "University of Kelaniya", "No"),
        ("Translation Studies", "098E", "University of Jaffna", "No"),
        ("Translation Studies", "098H", "Eastern University", "No"),
        ("Translation Studies", "098L", "Sabaragamuwa University", "No"),
        ("Software Engineering", "099C", "University of Sri Jayewardenepura", "No"),
        ("Software Engineering", "099D", "University of Kelaniya", "No"),
        ("Software Engineering", "099L", "Sabaragamuwa University", "No"),
        ("Film & Television Studies", "100D", "University of Kelaniya", "Yes"),
        ("Project Management", "101R", "University of Vavuniya", "No"),
        ("Data Science", "136L", "Sabaragamuwa University", "No"),
        ("Primary Education", "137A", "University of Colombo", "No"),
        ("Medical Imaging Technology", "138A", "University of Colombo", "No"),
        ("Polymer Science and Industrial Management", "139C", "University of Sri Jayewardenepura", "No"),
        ("Service Management", "140P", "Gampaha Wickramarachchi University", "No")
    ]
}

print("✅ Stream eligibility map loaded.")

# ------------------------------------------------------------
# 2. LOAD DATASET TO BUILD VERIFIED COURSE MAPPING (if available)
# ------------------------------------------------------------
dataset_path = "/content/ugc_al_admission_raw.csv"
verified_course_mapping = {}

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"✅ Dataset found, {len(df)} rows.")
    unique_courses = df[['Course', 'University', 'Uni Code', 'aptitude_required']].drop_duplicates()
    for _, row in unique_courses.iterrows():
        name = str(row['Course']).strip()
        upper = name.upper()
        verified_course_mapping[upper] = {
            'name': name,
            'university': str(row['University']),
            'uni_code': str(row['Uni Code']),
            'aptitude': str(row['aptitude_required'])
        }
    print(f"✅ Built mapping for {len(verified_course_mapping)} courses.")
else:
    # Build mapping from the eligibility map
    for stream, courses in STREAM_ELIGIBILITY.items():
        for c in courses:
            name, code, uni, apt = c
            upper = name.upper()
            verified_course_mapping[upper] = {
                'name': name,
                'university': uni,
                'uni_code': code,
                'aptitude': apt
            }
    print(f"✅ Created mapping from eligibility map ({len(verified_course_mapping)} courses).")

# ------------------------------------------------------------
# 3. HELPER DICTIONARIES (prestige, demand)
# ------------------------------------------------------------
UNIVERSITY_PRESTIGE = {
    "University of Colombo": 1.0,
    "University of Peradeniya": 0.98,
    "University of Moratuwa": 0.97,
    "University of Sri Jayewardenepura": 0.95,
    "University of Kelaniya": 0.92,
    "University of Jaffna": 0.88,
    "University of Ruhuna": 0.87,
    "Eastern University": 0.85,
    "South Eastern University": 0.83,
    "Rajarata University": 0.82,
    "Sabaragamuwa University": 0.81,
    "Wayamba University": 0.80,
    "Uva Wellassa University": 0.79,
    "Gampaha Wickramarachchi University": 0.78,
    "UCSC": 0.90,
    "Trincomalee Campus": 0.75,
    "University of Vavuniya": 0.74,
    "Sripalee Campus, University of Colombo": 0.76,
    "Swamy Vipulananda Institute": 0.77
}

COURSE_DEMAND = {
    "Medicine": 1.0,
    "Dental Surgery": 0.99,
    "Engineering": 0.98,
    "Computer Science": 0.97,
    "Software Engineering": 0.96,
    "Information Technology": 0.95,
    "Pharmacy": 0.94,
    "Nursing": 0.93,
    "Physiotherapy": 0.92,
    "Quantity Surveying": 0.91,
    "Management": 0.90,
    "Accountancy": 0.89,
    "Architecture": 0.88,
    "Law": 0.87,
    "Biological Science": 0.86,
    "Physical Science": 0.85,
    "Arts": 0.84,
    "Commerce": 0.83,
    "Engineering Technology": 0.82,
    "Biosystems Technology": 0.81
}
DEFAULT_DEMAND = 0.70

# ------------------------------------------------------------
# 4. FUNCTIONS TO FIND MATCHING COURSES AND CHECK STREAM
# ------------------------------------------------------------
def find_matching_courses(stream):
    """Return list of course details (name, university, code, aptitude) for a given stream."""
    if stream not in STREAM_ELIGIBILITY:
        return []
    eligible = STREAM_ELIGIBILITY[stream]
    matches = []
    seen = set()
    for course_data in eligible:
        name, code, uni, apt = course_data
        upper = name.upper()
        # Prefer verified mapping if available
        if upper in verified_course_mapping:
            details = verified_course_mapping[upper]
            key = f"{details['name']}_{details['university']}"
            if key not in seen:
                matches.append(details)
                seen.add(key)
        else:
            key = f"{name}_{uni}"
            if key not in seen:
                matches.append({
                    'name': name,
                    'university': uni,
                    'uni_code': code,
                    'aptitude': apt
                })
                seen.add(key)
    return matches

def course_belongs_to_stream(course_name, stream):
    """Check if a course name belongs to the given stream (exact or partial match)."""
    if stream not in STREAM_ELIGIBILITY:
        return False
    upper = course_name.upper()
    for c in STREAM_ELIGIBILITY[stream]:
        if c[0].upper() in upper or upper in c[0].upper():
            return True
    return False

# ------------------------------------------------------------
# 5. Q‑SCORE BIAS CALCULATION
# ------------------------------------------------------------
STREAM_Q_IMPORTANCE = {
    "Physical Science": {
        "q1_science_tech": 1.0, "q4_data": 0.9, "q8_hands_on": 0.8, "q9_innovation": 0.8,
        "q2_healthcare": 0.3, "q3_design": 0.4, "q5_business": 0.3, "q6_arts_culture": 0.1,
        "q7_nature_env": 0.3, "q10_people_social": 0.2, "q11_urban_corporate": 0.4, "q12_flexible_path": 0.5
    },
    "Biological Science": {
        "q2_healthcare": 1.0, "q7_nature_env": 0.9, "q10_people_social": 0.7, "q1_science_tech": 0.6,
        "q4_data": 0.4, "q8_hands_on": 0.5, "q3_design": 0.2, "q5_business": 0.2, "q6_arts_culture": 0.1,
        "q9_innovation": 0.5, "q11_urban_corporate": 0.2, "q12_flexible_path": 0.3
    },
    "Commerce": {
        "q5_business": 1.0, "q4_data": 0.9, "q11_urban_corporate": 0.8, "q10_people_social": 0.7,
        "q12_flexible_path": 0.6, "q1_science_tech": 0.3, "q2_healthcare": 0.2, "q3_design": 0.4,
        "q6_arts_culture": 0.3, "q7_nature_env": 0.2, "q8_hands_on": 0.3, "q9_innovation": 0.5
    },
    "Arts": {
        "q6_arts_culture": 1.0, "q3_design": 0.9, "q10_people_social": 0.8, "q12_flexible_path": 0.7,
        "q1_science_tech": 0.2, "q2_healthcare": 0.2, "q4_data": 0.3, "q5_business": 0.3,
        "q7_nature_env": 0.4, "q8_hands_on": 0.5, "q9_innovation": 0.6, "q11_urban_corporate": 0.4
    },
    "Engineering Technology": {
        "q1_science_tech": 1.0, "q8_hands_on": 0.9, "q9_innovation": 0.8, "q4_data": 0.7, "q3_design": 0.6,
        "q2_healthcare": 0.3, "q5_business": 0.4, "q6_arts_culture": 0.2, "q7_nature_env": 0.4,
        "q10_people_social": 0.3, "q11_urban_corporate": 0.5, "q12_flexible_path": 0.5
    },
    "Biosystems Technology": {
        "q7_nature_env": 1.0, "q2_healthcare": 0.8, "q1_science_tech": 0.7, "q8_hands_on": 0.6,
        "q3_design": 0.4, "q4_data": 0.4, "q5_business": 0.3, "q6_arts_culture": 0.2, "q9_innovation": 0.5,
        "q10_people_social": 0.4, "q11_urban_corporate": 0.3, "q12_flexible_path": 0.4
    }
}

def calculate_q_bias(stream, q_scores):
    if stream not in STREAM_Q_IMPORTANCE:
        return 1.0
    weights = STREAM_Q_IMPORTANCE[stream]
    total_score = total_weight = 0
    for q, w in weights.items():
        if q in q_scores:
            total_score += q_scores[q] * w
            total_weight += w
    if total_weight == 0:
        return 1.0
    avg = total_score / total_weight
    # bias factor in [0.8, 1.2]
    return 0.8 + avg / 25

def calculate_score(course_name, university, z_score, rank, q_scores, stream):
    # performance (30%)
    z_factor = min(1.0, z_score / 4.0)
    rank_factor = max(0.5, 1.0 - rank / 5000)
    perf = (z_factor * 0.6 + rank_factor * 0.4) * 0.3

    # prestige (25%)
    uni_up = university.upper()
    prest = 0.7 * 0.25
    for name, p in UNIVERSITY_PRESTIGE.items():
        if name.upper() in uni_up or uni_up in name.upper():
            prest = p * 0.25
            break

    # demand (25%)
    course_up = course_name.upper()
    dem = DEFAULT_DEMAND * 0.25
    for name, d in COURSE_DEMAND.items():
        if name.upper() in course_up or course_up in name.upper():
            dem = d * 0.25
            break

    # q‑bias (20%)
    q_bias = calculate_q_bias(stream, q_scores)
    q_part = q_bias * 0.20

    return perf + prest + dem + q_part

# ------------------------------------------------------------
# 6. MAIN RECOMMENDATION FUNCTION (with top_n validation 0‑30)
# ------------------------------------------------------------
def predict_courses(user_input, top_n=10, diversity=True):
    """
    Generate top‑N course recommendations (0 ≤ N ≤ 30).

    Parameters:
        user_input : dict with keys 'Stream', 'Z_Score', 'Island_Rank', 'District', and all q1..q12
        top_n      : number of recommendations to return (will be clipped to 0‑30)
        diversity  : if True, at most 2 courses from the same university

    Returns:
        dict with 'recommendations' (list of dicts) and 'input_summary'
    """
    # Validate and clip top_n to 0‑30
    if top_n < 0:
        top_n = 0
    elif top_n > 30:
        top_n = 30

    if top_n == 0:
        return {"recommendations": [], "input_summary": {}}

    stream = user_input.get("Stream", "")
    if stream not in STREAM_ELIGIBILITY:
        print(f"⚠️ Unknown stream: {stream}")
        return {"recommendations": [], "input_summary": {}}

    # gather q‑scores
    q_scores = {f"q{i}": user_input.get(f"q{i}", 3) for i in range(1,13)}
    # rename to full names used in bias dict
    q_full = {
        "q1_science_tech": q_scores["q1"],
        "q2_healthcare": q_scores["q2"],
        "q3_design": q_scores["q3"],
        "q4_data": q_scores["q4"],
        "q5_business": q_scores["q5"],
        "q6_arts_culture": q_scores["q6"],
        "q7_nature_env": q_scores["q7"],
        "q8_hands_on": q_scores["q8"],
        "q9_innovation": q_scores["q9"],
        "q10_people_social": q_scores["q10"],
        "q11_urban_corporate": q_scores["q11"],
        "q12_flexible_path": q_scores["q12"]
    }

    matching = find_matching_courses(stream)
    if not matching:
        print(f"⚠️ No courses found for {stream}")
        return {"recommendations": [], "input_summary": {}}

    z = user_input.get("Z_Score", 1.5)
    rank = user_input.get("Island_Rank", 1000)

    scored = []
    for c in matching:
        score = calculate_score(c['name'], c['university'], z, rank, q_full, stream)
        scored.append({
            'name': c['name'],
            'university': c['university'],
            'uni_code': c['uni_code'],
            'aptitude': c['aptitude'],
            'score': score
        })

    scored.sort(key=lambda x: x['score'], reverse=True)

    # diversity: at most 2 from same university
    recs = []
    uni_count = {}
    for c in scored:
        if len(recs) >= top_n:
            break
        uni = c['university']
        if diversity and uni_count.get(uni, 0) >= 2:
            continue
        recs.append({
            'rank': len(recs)+1,
            'course': c['name'],
            'score': round(c['score'], 4),
            'university': c['university'],
            'uni_code': c['uni_code'],
            'aptitude_required': c['aptitude']
        })
        uni_count[uni] = uni_count.get(uni, 0) + 1

    # if diversity left out courses, fill with highest remaining
    if len(recs) < top_n:
        for c in scored:
            if any(r['course'] == c['name'] for r in recs):
                continue
            if len(recs) >= top_n:
                break
            recs.append({
                'rank': len(recs)+1,
                'course': c['name'],
                'score': round(c['score'], 4),
                'university': c['university'],
                'uni_code': c['uni_code'],
                'aptitude_required': c['aptitude']
            })

    return {
        'recommendations': recs,
        'input_summary': {
            'stream': stream,
            'z_score': z,
            'island_rank': rank,
            'district': user_input.get('District', 'Unknown')
        }
    }

# ------------------------------------------------------------
# 7. TEST CASES (from your previous code)
# ------------------------------------------------------------
test_cases_json = [
    {
        "test_id": 1,
        "name": "Physical Science - Top Performer",
        "category": "Good",
        "stream": "Physical Science",
        "input": {
            "Year": 2024, "Stream": "Physical Science",
            "Subject_1": "Physics", "Grade_1": "A",
            "Subject_2": "Chemistry", "Grade_2": "A",
            "Subject_3": "Combined Mathematics", "Grade_3": "A",
            "Z_Score": 3.2, "Island_Rank": 15, "District": "Colombo", "Gen_Test": 95,
            "Sinhala/Tamil": "A", "English": "A", "Maths": "A", "Science": "A",
            "q1_science_tech": 5, "q2_healthcare": 2, "q3_design": 3, "q4_data": 5,
            "q5_business": 1, "q6_arts_culture": 1, "q7_nature_env": 2, "q8_hands_on": 5,
            "q9_innovation": 5, "q10_people_social": 2, "q11_urban_corporate": 4, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 2,
        "name": "Biological Science - Medical Aspirant",
        "category": "Good",
        "stream": "Biological Science",
        "input": {
            "Year": 2024, "Stream": "Biological Science",
            "Subject_1": "Biology", "Grade_1": "A",
            "Subject_2": "Chemistry", "Grade_2": "A",
            "Subject_3": "Physics", "Grade_3": "A",
            "Z_Score": 3.0, "Island_Rank": 25, "District": "Kandy", "Gen_Test": 92,
            "Sinhala/Tamil": "A", "English": "A", "Maths": "A", "Science": "A",
            "q1_science_tech": 4, "q2_healthcare": 5, "q3_design": 2, "q4_data": 3,
            "q5_business": 1, "q6_arts_culture": 1, "q7_nature_env": 5, "q8_hands_on": 4,
            "q9_innovation": 4, "q10_people_social": 5, "q11_urban_corporate": 2, "q12_flexible_path": 2
        }
    },
    {
        "test_id": 3,
        "name": "Commerce - Top Management",
        "category": "Good",
        "stream": "Commerce",
        "input": {
            "Year": 2024, "Stream": "Commerce",
            "Subject_1": "Economics", "Grade_1": "A",
            "Subject_2": "Business Studies", "Grade_2": "A",
            "Subject_3": "Accounting", "Grade_3": "A",
            "Z_Score": 2.9, "Island_Rank": 40, "District": "Colombo", "Gen_Test": 90,
            "Sinhala/Tamil": "A", "English": "A", "Maths": "A", "Science": "B",
            "q1_science_tech": 2, "q2_healthcare": 1, "q3_design": 3, "q4_data": 5,
            "q5_business": 5, "q6_arts_culture": 2, "q7_nature_env": 1, "q8_hands_on": 3,
            "q9_innovation": 4, "q10_people_social": 4, "q11_urban_corporate": 5, "q12_flexible_path": 4
        }
    },
    {
        "test_id": 4,
        "name": "Arts - Top Humanities",
        "category": "Good",
        "stream": "Arts",
        "input": {
            "Year": 2024, "Stream": "Arts",
            "Subject_1": "Political Science", "Grade_1": "A",
            "Subject_2": "History", "Grade_2": "A",
            "Subject_3": "Geography", "Grade_3": "A",
            "Z_Score": 2.7, "Island_Rank": 80, "District": "Kandy", "Gen_Test": 88,
            "Sinhala/Tamil": "A", "English": "A", "Maths": "B", "Science": "B",
            "q1_science_tech": 1, "q2_healthcare": 2, "q3_design": 5, "q4_data": 2,
            "q5_business": 3, "q6_arts_culture": 5, "q7_nature_env": 3, "q8_hands_on": 2,
            "q9_innovation": 4, "q10_people_social": 5, "q11_urban_corporate": 3, "q12_flexible_path": 5
        }
    },
    {
        "test_id": 5,
        "name": "Engineering Technology - Top Engineer",
        "category": "Good",
        "stream": "Engineering Technology",
        "input": {
            "Year": 2024, "Stream": "Engineering Technology",
            "Subject_1": "Engineering Technology", "Grade_1": "A",
            "Subject_2": "Information Technology", "Grade_2": "A",
            "Subject_3": "Science for Technology", "Grade_3": "A",
            "Z_Score": 2.8, "Island_Rank": 60, "District": "Colombo", "Gen_Test": 89,
            "Sinhala/Tamil": "A", "English": "A", "Maths": "A", "Science": "A",
            "q1_science_tech": 5, "q2_healthcare": 2, "q3_design": 4, "q4_data": 4,
            "q5_business": 2, "q6_arts_culture": 1, "q7_nature_env": 2, "q8_hands_on": 5,
            "q9_innovation": 5, "q10_people_social": 2, "q11_urban_corporate": 3, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 6,
        "name": "Physical Science - Above Average",
        "category": "Above Average",
        "stream": "Physical Science",
        "input": {
            "Year": 2024, "Stream": "Physical Science",
            "Subject_1": "Physics", "Grade_1": "A",
            "Subject_2": "Chemistry", "Grade_2": "B",
            "Subject_3": "Mathematics", "Grade_3": "A",
            "Z_Score": 2.4, "Island_Rank": 180, "District": "Galle", "Gen_Test": 78,
            "Sinhala/Tamil": "A", "English": "B", "Maths": "A", "Science": "A",
            "q1_science_tech": 4, "q2_healthcare": 3, "q3_design": 3, "q4_data": 4,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 3, "q8_hands_on": 3,
            "q9_innovation": 4, "q10_people_social": 3, "q11_urban_corporate": 3, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 7,
        "name": "Biological Science - Above Average",
        "category": "Above Average",
        "stream": "Biological Science",
        "input": {
            "Year": 2024, "Stream": "Biological Science",
            "Subject_1": "Biology", "Grade_1": "A",
            "Subject_2": "Chemistry", "Grade_2": "B",
            "Subject_3": "Physics", "Grade_3": "B",
            "Z_Score": 2.3, "Island_Rank": 220, "District": "Matara", "Gen_Test": 76,
            "Sinhala/Tamil": "A", "English": "B", "Maths": "B", "Science": "A",
            "q1_science_tech": 3, "q2_healthcare": 4, "q3_design": 2, "q4_data": 3,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 4, "q8_hands_on": 3,
            "q9_innovation": 3, "q10_people_social": 4, "q11_urban_corporate": 2, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 8,
        "name": "Commerce - Above Average",
        "category": "Above Average",
        "stream": "Commerce",
        "input": {
            "Year": 2024, "Stream": "Commerce",
            "Subject_1": "Economics", "Grade_1": "A",
            "Subject_2": "Business Studies", "Grade_2": "B",
            "Subject_3": "Accounting", "Grade_3": "B",
            "Z_Score": 2.2, "Island_Rank": 280, "District": "Kurunegala", "Gen_Test": 74,
            "Sinhala/Tamil": "B", "English": "A", "Maths": "B", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 3, "q4_data": 4,
            "q5_business": 4, "q6_arts_culture": 3, "q7_nature_env": 2, "q8_hands_on": 3,
            "q9_innovation": 3, "q10_people_social": 3, "q11_urban_corporate": 4, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 9,
        "name": "Arts - Average",
        "category": "Average",
        "stream": "Arts",
        "input": {
            "Year": 2024, "Stream": "Arts",
            "Subject_1": "History", "Grade_1": "B",
            "Subject_2": "Geography", "Grade_2": "C",
            "Subject_3": "Political Science", "Grade_3": "B",
            "Z_Score": 1.5, "Island_Rank": 820, "District": "Kurunegala", "Gen_Test": 58,
            "Sinhala/Tamil": "B", "English": "C", "Maths": "C", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 4, "q4_data": 2,
            "q5_business": 3, "q6_arts_culture": 4, "q7_nature_env": 3, "q8_hands_on": 2,
            "q9_innovation": 3, "q10_people_social": 4, "q11_urban_corporate": 2, "q12_flexible_path": 4
        }
    },
    {
        "test_id": 10,
        "name": "Engineering Technology - Average",
        "category": "Average",
        "stream": "Engineering Technology",
        "input": {
            "Year": 2024, "Stream": "Engineering Technology",
            "Subject_1": "Engineering Technology", "Grade_1": "B",
            "Subject_2": "Information Technology", "Grade_2": "C",
            "Subject_3": "Science for Technology", "Grade_3": "B",
            "Z_Score": 1.6, "Island_Rank": 750, "District": "Galle", "Gen_Test": 61,
            "Sinhala/Tamil": "B", "English": "C", "Maths": "B", "Science": "B",
            "q1_science_tech": 4, "q2_healthcare": 2, "q3_design": 3, "q4_data": 4,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 2, "q8_hands_on": 4,
            "q9_innovation": 3, "q10_people_social": 2, "q11_urban_corporate": 3, "q12_flexible_path": 3
        }
    },
    {
        "test_id": 11,
        "name": "Physical Science - Below Average",
        "category": "Below Average",
        "stream": "Physical Science",
        "input": {
            "Year": 2024, "Stream": "Physical Science",
            "Subject_1": "Physics", "Grade_1": "C",
            "Subject_2": "Chemistry", "Grade_2": "C",
            "Subject_3": "Mathematics", "Grade_3": "C",
            "Z_Score": 1.2, "Island_Rank": 1200, "District": "Anuradhapura", "Gen_Test": 48,
            "Sinhala/Tamil": "C", "English": "C", "Maths": "C", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 2, "q4_data": 2,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 2, "q8_hands_on": 2,
            "q9_innovation": 2, "q10_people_social": 2, "q11_urban_corporate": 2, "q12_flexible_path": 2
        }
    },
    {
        "test_id": 12,
        "name": "Biological Science - Below Average",
        "category": "Below Average",
        "stream": "Biological Science",
        "input": {
            "Year": 2024, "Stream": "Biological Science",
            "Subject_1": "Biology", "Grade_1": "C",
            "Subject_2": "Chemistry", "Grade_2": "C",
            "Subject_3": "Physics", "Grade_3": "C",
            "Z_Score": 1.1, "Island_Rank": 1350, "District": "Badulla", "Gen_Test": 45,
            "Sinhala/Tamil": "C", "English": "C", "Maths": "C", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 2, "q4_data": 2,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 2, "q8_hands_on": 2,
            "q9_innovation": 2, "q10_people_social": 2, "q11_urban_corporate": 2, "q12_flexible_path": 2
        }
    },
    {
        "test_id": 13,
        "name": "Commerce - Below Average",
        "category": "Below Average",
        "stream": "Commerce",
        "input": {
            "Year": 2024, "Stream": "Commerce",
            "Subject_1": "Economics", "Grade_1": "C",
            "Subject_2": "Business Studies", "Grade_2": "C",
            "Subject_3": "Accounting", "Grade_3": "C",
            "Z_Score": 1.0, "Island_Rank": 1500, "District": "Jaffna", "Gen_Test": 42,
            "Sinhala/Tamil": "C", "English": "C", "Maths": "C", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 2, "q4_data": 2,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 2, "q8_hands_on": 2,
            "q9_innovation": 2, "q10_people_social": 2, "q11_urban_corporate": 2, "q12_flexible_path": 2
        }
    },
    {
        "test_id": 14,
        "name": "Arts - Below Average",
        "category": "Below Average",
        "stream": "Arts",
        "input": {
            "Year": 2024, "Stream": "Arts",
            "Subject_1": "History", "Grade_1": "C",
            "Subject_2": "Geography", "Grade_2": "C",
            "Subject_3": "Political Science", "Grade_3": "C",
            "Z_Score": 0.9, "Island_Rank": 1650, "District": "Matara", "Gen_Test": 40,
            "Sinhala/Tamil": "C", "English": "C", "Maths": "C", "Science": "C",
            "q1_science_tech": 2, "q2_healthcare": 2, "q3_design": 2, "q4_data": 2,
            "q5_business": 2, "q6_arts_culture": 2, "q7_nature_env": 2, "q8_hands_on": 2,
            "q9_innovation": 2, "q10_people_social": 2, "q11_urban_corporate": 2, "q12_flexible_path": 2
        }
    }
]

# ------------------------------------------------------------
# 8. RUN ALL TEST CASES AND DISPLAY RESULTS
# ------------------------------------------------------------
print("\n" + "="*80)
print("RUNNING TEST CASES – Q‑SCORE BIASED RECOMMENDATIONS")
print("="*80)

all_results = []

for test in test_cases_json:
    print("\n" + "="*80)
    print(f"TEST CASE #{test['test_id']}: {test['name']}")
    print(f"Category: {test['category']}")
    print("="*80)

    inp = test['input']
    print(f"\n📋 INPUT SUMMARY:")
    print(f"   Stream: {test['stream']}")
    print(f"   Z-Score: {inp['Z_Score']}")
    print(f"   Island Rank: {inp['Island_Rank']}")
    print(f"   District: {inp['District']}")
    print(f"   Q-Scores: Science/Tech={inp['q1_science_tech']}, Healthcare={inp['q2_healthcare']}, Design={inp['q3_design']}")

    result = predict_courses(inp, top_n=10)

    if result['recommendations']:
        print(f"\n📊 TOP RECOMMENDATIONS FOR {test['stream'].upper()}:")
        rec_table = []
        for rec in result['recommendations']:
            course_short = rec['course'][:35] + "..." if len(rec['course']) > 35 else rec['course']
            uni_short = rec['university'][:25] + "..." if len(rec['university']) > 25 else rec['university']
            rec_table.append([
                rec['rank'],
                course_short,
                uni_short,
                rec['uni_code'],
                rec['aptitude_required'],
                f"{rec['score']:.4f}"
            ])
        print(tabulate(rec_table,
                       headers=['Rank', 'Course', 'University', 'Code', 'Aptitude', 'Score'],
                       tablefmt='grid', maxcolwidths=[6,40,30,10,10,8]))

        # Verify each recommendation belongs to the stream
        print("\n🔍 VERIFICATION:")
        all_valid = True
        for rec in result['recommendations']:
            if course_belongs_to_stream(rec['course'], test['stream']):
                print(f"   ✅ {rec['course'][:40]}... → Valid")
            else:
                print(f"   ❌ {rec['course'][:40]}... → INVALID")
                all_valid = False
        if all_valid:
            print(f"\n✅ ALL RECOMMENDATIONS VALID for {test['stream']} stream")
    else:
        print("\n❌ No recommendations generated")

    all_results.append({
        "test_id": test['test_id'],
        "name": test['name'],
        "category": test['category'],
        "stream": test['stream'],
        "recommendations": result['recommendations']
    })
    print("\n" + "-"*80)

# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------
with open('test_results_qbiased.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("\n✅ Test results saved to 'test_results_qbiased.json'")

# ------------------------------------------------------------
# 10. DEMONSTRATION OF DIFFERENT top_n VALUES (0‑30)
# ------------------------------------------------------------
print("\n" + "="*80)
print("DEMONSTRATION: VARYING NUMBER OF RECOMMENDATIONS (top_n)")
print("="*80)

demo_test = test_cases_json[0]  # Physical Science - Top Performer
demo_input = demo_test['input']
print(f"\nUsing test case: {demo_test['name']}")

for n in [5, 15, 25, 30]:
    print(f"\n--- top_n = {n} ---")
    res = predict_courses(demo_input, top_n=n)
    if res['recommendations']:
        print(f"Returned {len(res['recommendations'])} recommendations")
        # print first few as sample
        for r in res['recommendations'][:3]:
            print(f"  Rank {r['rank']}: {r['course']} ({r['university']}) - Score {r['score']:.4f}")
    else:
        print("No recommendations")

print("\n" + "="*80)
print("✅ ALL TESTS COMPLETED – Q‑SCORE BIASED RECOMMENDATIONS")
print("="*80)

✅ Stream eligibility map loaded.
✅ Dataset found, 3768 rows.
✅ Built mapping for 153 courses.

RUNNING TEST CASES – Q‑SCORE BIASED RECOMMENDATIONS

TEST CASE #1: Physical Science - Top Performer
Category: Good

📋 INPUT SUMMARY:
   Stream: Physical Science
   Z-Score: 3.2
   Island Rank: 15
   District: Colombo
   Q-Scores: Science/Tech=5, Healthcare=2, Design=3

📊 TOP RECOMMENDATIONS FOR PHYSICAL SCIENCE:
+--------+---------------------------------+------------------------------+--------+------------+---------+
|   Rank | Course                          | University                   | Code   | Aptitude   |   Score |
+========+=================================+==============================+========+============+=========+
|      1 | Engineering                     | University of Peradeniya     | 013B   | No         |  0.9376 |
+--------+---------------------------------+------------------------------+--------+------------+---------+
|      2 | ENGINEERING (EM)                | UNIVER

**Cell 18: (Optional) Save test results for later inspection**

In [44]:
def get_recommendations(student_data, model, encoders, scaler, feature_columns):

    import pandas as pd

    # Convert to DataFrame
    df_input = pd.DataFrame([student_data])

    # Encode categorical variables
    for col, encoder in encoders.items():
        if col in df_input.columns:
            df_input[col] = encoder.transform(df_input[col])

    # Ensure column order
    df_input = df_input[feature_columns]

    # Scale numerical data
    X_scaled = scaler.transform(df_input)

    # Predict probabilities
    probs = model.predict_proba(X_scaled)[0]

    # Get top 5 courses
    top_indices = probs.argsort()[-5:][::-1]

    recommendations = []
    for idx in top_indices:
        recommendations.append({
            "course": model.classes_[idx],
            "probability": float(probs[idx])
        })

    return recommendations